<a href="https://colab.research.google.com/github/raw-fun/Colab-Script/blob/main/Boss.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 01 — MASTER FOUNDATION
# Package install + imports + Enums + Config + Helpers
# ══════════════════════════════════════════════════════════════

import subprocess
subprocess.run(
    ["pip", "install", "-q",
     "Pillow>=10.0.0,<12.0.0", "qrcode[pil]",
     "fpdf2", "ipywidgets", "tqdm"],
    capture_output=True
)

import os, sys, io, json, csv, zipfile, textwrap
import time, math, random, base64, hashlib
import urllib.request, urllib.error
import dataclasses
import inspect
import builtins
import traceback
from pathlib import Path
from datetime import datetime
from typing import Optional, Dict, List, Tuple, Any, Union
from dataclasses import dataclass, field, asdict
from enum import Enum
from collections import defaultdict

import numpy as np
import cv2
from PIL import (
    Image, ImageDraw, ImageFont,
    ImageFilter, ImageEnhance, ImageChops, ImageColor
)
import qrcode

try:
    from fpdf import FPDF
    FPDF_AVAILABLE = True
except ImportError:
    FPDF_AVAILABLE = False

try:
    from tqdm.notebook import tqdm
except ImportError:
    class tqdm:
        def __init__(self, iterable=None, total=None, desc="", **kw):
            self.iterable = iterable or []
            self.n = 0
        def __iter__(self):
            for x in self.iterable:
                yield x; self.n += 1
        def __enter__(self): return self
        def __exit__(self, *a): pass
        def update(self, n=1): self.n += n
        def set_postfix(self, d=None, **kw): pass

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# ── Availability flags ────────────────────────────────────────
REMBG_AVAILABLE   = False
SKLEARN_AVAILABLE = False
CV2_AVAILABLE     = True  # cv2 already imported above

try:
    import rembg as _rembg
    REMBG_AVAILABLE = True
except ImportError:
    pass
try:
    import sklearn as _sk
    SKLEARN_AVAILABLE = True
except ImportError:
    pass

# Global scope — সব cell থেকে access করা যাবে
builtins.REMBG_AVAILABLE    = REMBG_AVAILABLE
builtins.SKLEARN_AVAILABLE  = SKLEARN_AVAILABLE
builtins.CV2_AVAILABLE      = CV2_AVAILABLE
builtins.FPDF_AVAILABLE     = FPDF_AVAILABLE

print(f"✅ NumPy {np.__version__} | Pillow {Image.__version__} | OpenCV {cv2.__version__}")

# ══════════════════════════════════════════════════════════════
# SECTION A — ENUMS
# ══════════════════════════════════════════════════════════════

class GradientDirection(Enum):
    VERTICAL   = "vertical"
    HORIZONTAL = "horizontal"
    DIAGONAL   = "diagonal"
    RADIAL     = "radial"

class TextureType(Enum):
    NONE      = "none"
    PAPER     = "paper"
    GRAIN     = "grain"
    GEOMETRIC = "geometric"
    LINES     = "lines"
    DOTS      = "dots"

class ImageShape(Enum):
    CIRCLE         = "circle"
    STAR           = "star"
    HEXAGON        = "hexagon"
    HEART          = "heart"
    ROUNDED_SQUARE = "rounded_square"
    DIAMOND        = "diamond"
    TRIANGLE       = "triangle"

class QRStyle(Enum):
    SQUARE  = "square"
    ROUNDED = "rounded"
    DOTS    = "dots"

class TextEffect(Enum):
    NONE     = "none"
    GRADIENT = "gradient"
    NEON     = "neon"
    OUTLINED = "outlined"
    EMBOSSED = "embossed"
    SHADOW   = "shadow"

# ══════════════════════════════════════════════════════════════
# SECTION B — CONFIG
# ══════════════════════════════════════════════════════════════

@dataclass
class Config:
    # Directories
    FONT_DIR:        str  = "fonts"
    OUTPUT_DIR:      str  = "outputs"
    EXPORT_DIR:      str  = "exports"
    BULK_DIR:        str  = "bulk"
    BRAND_DIR:       str  = "brand_kits"
    ASSET_DIR:       str  = "assets"
    CACHE_DIR:       str  = "cache"
    TEMP_DIR:        str  = "temp"
    # Canvas
    DEFAULT_WIDTH:   int  = 1080
    DEFAULT_HEIGHT:  int  = 1080
    DEFAULT_DPI:     int  = 96
    MAX_CANVAS_SIZE: int  = 5000
    # Quality
    JPEG_QUALITY:    int  = 93
    WEBP_QUALITY:    int  = 88
    # Processing
    MAX_WORKERS:     int  = 2
    CHUNK_SIZE:      int  = 10
    MAX_BULK:        int  = 500
    RETRY_COUNT:     int  = 3
    TIMEOUT:         int  = 30
    THREAD_POOL:     int  = 2
    # Features
    CACHE_ENABLED:   bool = True
    PROGRESS_BAR:    bool = True
    AUTO_SAVE:       bool = True
    TIMESTAMP_FILES: bool = True
    WATERMARK:       bool = False
    WATERMARK_TEXT:  str  = "ColabCanvas"
    OUTPUT_PREFIX:   str  = "bulk_"
    REPORT_FORMAT:   str  = "json"
    ZIP_COMPRESSION: int  = 6
    PREVIEW_SIZE:    int  = 400
    LOG_LEVEL:       str  = "INFO"

    def create_dirs(self):
        for d in [self.FONT_DIR, self.OUTPUT_DIR,
                  self.EXPORT_DIR, self.BULK_DIR,
                  self.BRAND_DIR, self.ASSET_DIR,
                  self.CACHE_DIR, self.TEMP_DIR]:
            os.makedirs(d, exist_ok=True)

CONFIG = Config()
CONFIG.create_dirs()

# ══════════════════════════════════════════════════════════════
# SECTION C — CONSTANTS
# ══════════════════════════════════════════════════════════════

FONT_SCALE = {
    "display":     96,  "display_sm":  80,  "display_md":  96,
    "display_lg": 120,  "display_xl": 144,
    "h1": 80, "h2": 64, "h3": 52, "h4": 42, "h5": 34,
    "h1_sm": 64, "h2_sm": 52, "h3_sm": 42,
    "h1_lg": 96, "h2_lg": 80, "h3_lg": 64,
    "subtitle": 30, "body_lg": 26, "body": 22,
    "body_sm": 18, "caption": 16, "micro": 13,
    "xl": 80, "lg": 64, "md": 52, "sm": 34, "xs": 22,
    "2xs": 16, "3xs": 13, "xxl": 100, "giant": 140,
    "huge": 120, "hero": 110, "jumbo": 130, "tiny": 12,
    "price": 88, "price_lg": 110, "price_sm": 64,
    "tag": 48, "tag_sm": 36, "tag_lg": 60,
    "label": 24, "label_sm": 18, "label_lg": 32,
    "overline": 18, "eyebrow": 20,
    "numeral": 100, "numeral_sm": 72, "numeral_lg": 130,
    "bengali_hero": 100, "bengali_h1": 80,
    "bengali_h2": 64, "bengali_body": 28,
}

SPACING = {
    "xs": 8, "sm": 16, "md": 24,
    "lg": 40, "xl": 60, "2xl": 80, "3xl": 120,
}

LAYOUT = {
    "instagram_post":  {"w": 1080, "h": 1080, "margin": 80},
    "instagram_story": {"w": 1080, "h": 1920, "margin": 80},
    "facebook_cover":  {"w": 1640, "h":  624, "margin": 60},
    "youtube_thumb":   {"w": 1280, "h":  720, "margin": 60},
    "twitter_header":  {"w": 1500, "h":  500, "margin": 60},
    "linkedin_banner": {"w": 1584, "h":  396, "margin": 60},
    "a4_portrait":     {"w": 2480, "h": 3508, "margin": 120},
    "a4_landscape":    {"w": 3508, "h": 2480, "margin": 120},
    "square_sm":       {"w":  800, "h":  800, "margin": 60},
    "wide_sm":         {"w": 1200, "h":  628, "margin": 60},
}

GRADIENT_PRESETS = {
    "sunset":   ("#FF6B6B", "#FFA726"),
    "ocean":    ("#0F2027", "#2C5364"),
    "purple":   ("#6A0572", "#AB83A1"),
    "neon":     ("#00F0FF", "#6366F1"),
    "gold":     ("#F6D365", "#FDA085"),
    "forest":   ("#11998E", "#38EF7D"),
    "fire":     ("#F7971E", "#FFD200"),
    "midnight": ("#0F0C29", "#302B63"),
    "rose":     ("#F953C6", "#B91D73"),
    "royal":    ("#141E30", "#243B55"),
}

EFFECT_PRESETS = {
    "none":      [],
    "minimal":   ["vignette"],
    "cinematic": ["vignette", "grain", "color_grade"],
    "neon":      ["bokeh", "vignette", "grain"],
    "elegant":   ["vignette", "texture"],
    "vibrant":   ["bokeh", "vignette"],
}

# Global scope — সব পরবর্তী cell-এ পাওয়া যাবে
builtins.FONT_SCALE       = FONT_SCALE
builtins.SPACING          = SPACING
builtins.LAYOUT           = LAYOUT
builtins.GRADIENT_PRESETS = GRADIENT_PRESETS
builtins.EFFECT_PRESETS   = EFFECT_PRESETS

# ══════════════════════════════════════════════════════════════
# SECTION D — HELPER FUNCTIONS
# ══════════════════════════════════════════════════════════════

def clamp(val, lo, hi):
    return max(lo, min(hi, val))

def safe_int(val, default: int = 0) -> int:
    """'center'/'right'/'left'/None → default; অন্যথা int()"""
    if val is None: return default
    if isinstance(val, int): return val
    if isinstance(val, float): return int(val)
    if isinstance(val, str):
        v = val.strip().lower()
        if v in ('center','centre','left','right','top','bottom','middle'):
            return default
        try: return int(v)
        except: return default
    return default

def hex_to_rgb(hex_color: str) -> Tuple[int, int, int]:
    h = hex_color.strip().lstrip('#')
    if len(h) == 3:
        h = ''.join(c*2 for c in h)
    return tuple(int(h[i:i+2], 16) for i in (0, 2, 4))

def rgb_to_hex(r: int, g: int, b: int) -> str:
    return f"#{int(r):02x}{int(g):02x}{int(b):02x}"

def hex_with_alpha(hex_color: str, alpha: float) -> Tuple:
    r, g, b = hex_to_rgb(hex_color)
    return (r, g, b, int(255 * clamp(alpha, 0, 1)))

def blend_colors(c1: str, c2: str, t: float) -> str:
    r1,g1,b1 = hex_to_rgb(c1)
    r2,g2,b2 = hex_to_rgb(c2)
    t = clamp(t, 0, 1)
    return rgb_to_hex(
        int(r1+(r2-r1)*t),
        int(g1+(g2-g1)*t),
        int(b1+(b2-b1)*t)
    )

def interpolate_color(c1: str, c2: str, ratio: float) -> tuple:
    ratio = clamp(ratio, 0.0, 1.0)
    r1,g1,b1 = hex_to_rgb(c1)
    r2,g2,b2 = hex_to_rgb(c2)
    return (
        int(r1+(r2-r1)*ratio),
        int(g1+(g2-g1)*ratio),
        int(b1+(b2-b1)*ratio)
    )

def luminance(hex_color: str) -> float:
    def lin(c):
        c /= 255
        return c/12.92 if c <= 0.04045 else ((c+0.055)/1.055)**2.4
    r,g,b = hex_to_rgb(hex_color)
    return 0.2126*lin(r) + 0.7152*lin(g) + 0.0722*lin(b)

def contrast_ratio(c1: str, c2: str) -> float:
    l1,l2 = luminance(c1), luminance(c2)
    hi,lo = max(l1,l2), min(l1,l2)
    return (hi+0.05)/(lo+0.05)

def auto_text_color(bg: str) -> str:
    return "#FFFFFF" if luminance(bg) < 0.4 else "#111111"

def make_gradient_image(
    w: int, h: int,
    c1: str, c2: str,
    direction: str = "horizontal"
) -> Image.Image:
    arr = np.zeros((h, w, 4), dtype=np.uint8)
    r1,g1,b1 = hex_to_rgb(c1)
    r2,g2,b2 = hex_to_rgb(c2)
    if direction == "vertical":
        t = np.tile(np.linspace(0,1,h)[:,None], (1,w))
    elif direction == "diagonal":
        row_t = np.linspace(0,1,h)[:,None]
        col_t = np.linspace(0,1,w)[None,:]
        t     = (row_t + col_t) / 2
    else:  # horizontal
        t = np.tile(np.linspace(0,1,w)[None,:], (h,1))
    arr[:,:,0] = (r1+(r2-r1)*t).astype(np.uint8)
    arr[:,:,1] = (g1+(g2-g1)*t).astype(np.uint8)
    arr[:,:,2] = (b1+(b2-b1)*t).astype(np.uint8)
    arr[:,:,3] = 255
    return Image.fromarray(arr, "RGBA")

def extract_dominant_colors(
    image_bytes: bytes, n_colors: int = 5
) -> List[str]:
    img    = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    img    = img.resize((100, 100), Image.Resampling.LANCZOS)
    pixels = np.array(img).reshape(-1, 3).astype(np.float32)
    rng    = np.random.default_rng(42)
    centers = pixels[rng.choice(len(pixels), n_colors, replace=False)]
    for _ in range(100):
        diff   = pixels[:,None,:] - centers[None,:,:]
        dists  = np.einsum('nkd,nkd->nk', diff, diff)
        labels = np.argmin(dists, axis=1)
        new_c  = np.array([
            pixels[labels==k].mean(axis=0)
            if np.any(labels==k) else centers[k]
            for k in range(n_colors)
        ])
        if np.allclose(centers, new_c, atol=0.5):
            break
        centers = new_c
    return [rgb_to_hex(*c.astype(int)) for c in centers]

def remove_background(img_input) -> Image.Image:
    """OpenCV GrabCut background removal"""
    try:
        if isinstance(img_input, bytes):
            arr    = np.frombuffer(img_input, np.uint8)
            img_cv = cv2.imdecode(arr, cv2.IMREAD_COLOR)
        elif isinstance(img_input, Image.Image):
            img_cv = cv2.cvtColor(
                np.array(img_input.convert("RGB")), cv2.COLOR_RGB2BGR)
        else:
            img_cv = img_input

        h, w      = img_cv.shape[:2]
        mask      = np.zeros((h, w), np.uint8)
        bgd_model = np.zeros((1,65), np.float64)
        fgd_model = np.zeros((1,65), np.float64)
        m         = max(10, min(h,w)//10)
        rect      = (m, m, w-2*m, h-2*m)
        cv2.grabCut(img_cv, mask, rect, bgd_model, fgd_model,
                    5, cv2.GC_INIT_WITH_RECT)
        fg = np.where((mask==2)|(mask==0), 0, 255).astype(np.uint8)
        fg = cv2.GaussianBlur(fg, (5,5), 0)
        _, fg = cv2.threshold(fg, 127, 255, cv2.THRESH_BINARY)
        rgb = cv2.cvtColor(img_cv, cv2.COLOR_BGR2RGB)
        return Image.fromarray(np.dstack([rgb, fg]), 'RGBA')
    except Exception:
        if isinstance(img_input, bytes):
            return Image.open(io.BytesIO(img_input)).convert("RGBA")
        return (img_input.convert("RGBA")
                if hasattr(img_input, 'convert')
                else Image.new("RGBA", (200,200), (0,0,0,0)))

# Global helpers
builtins.safe_int            = safe_int
builtins.interpolate_color   = interpolate_color
builtins.make_gradient_image = make_gradient_image
builtins.auto_text_color     = auto_text_color
builtins.blend_colors        = blend_colors
builtins.extract_dominant_colors = extract_dominant_colors
builtins.remove_background   = remove_background

print("✅ Enums, Config, Constants, Helpers — সব ready!")
print(f"   CONFIG: {len(dataclasses.fields(CONFIG))} fields")
print(f"   GRADIENT_PRESETS: {len(GRADIENT_PRESETS)}")
print(f"   EFFECT_PRESETS:   {len(EFFECT_PRESETS)}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 02 — FONT SYSTEM
# Download + Cache + FontManager
# ══════════════════════════════════════════════════════════════

FONT_CATALOG = {
    "bengali_bold": {
        "file": "bengali_bold.ttf",
        "url":  "https://github.com/google/fonts/raw/main/ofl/hindsiliguri/HindSiliguri-Bold.ttf",
    },
    "bengali_semibold": {
        "file": "bengali_semibold.ttf",
        "url":  "https://github.com/google/fonts/raw/main/ofl/hindsiliguri/HindSiliguri-SemiBold.ttf",
    },
    "bengali_regular": {
        "file": "bengali_regular.ttf",
        "url":  "https://github.com/google/fonts/raw/main/ofl/hindsiliguri/HindSiliguri-Regular.ttf",
    },
    "bengali_light": {
        "file": "bengali_light.ttf",
        "url":  "https://github.com/google/fonts/raw/main/ofl/hindsiliguri/HindSiliguri-Light.ttf",
    },
    "english_bold": {
        "file": "english_bold.ttf",
        "url":  "https://github.com/JulietaUla/Montserrat/raw/master/fonts/ttf/Montserrat-Bold.ttf",
        "fallback": "https://github.com/rsms/inter/raw/master/docs/font-files/Inter-Regular.ttf",
    },
    "english_semibold": {
        "file": "english_semibold.ttf",
        "url":  "https://github.com/JulietaUla/Montserrat/raw/master/fonts/ttf/Montserrat-SemiBold.ttf",
        "fallback": "https://github.com/rsms/inter/raw/master/docs/font-files/Inter-Regular.ttf",
    },
    "english_regular": {
        "file": "english_regular.ttf",
        "url":  "https://github.com/rsms/inter/raw/master/docs/font-files/Inter-Regular.ttf",
    },
    "english_light": {
        "file": "english_light.ttf",
        "url":  "https://github.com/rsms/inter/raw/master/docs/font-files/Inter-Light.ttf",
        "fallback": "https://github.com/rsms/inter/raw/master/docs/font-files/Inter-Regular.ttf",
    },
    "english_italic": {
        "file": "english_italic.ttf",
        "url":  "https://github.com/rsms/inter/raw/master/docs/font-files/Inter-Italic.ttf",
        "fallback": "https://github.com/rsms/inter/raw/master/docs/font-files/Inter-Regular.ttf",
    },
}

def _dl_font(url: str, dest: str) -> bool:
    if os.path.exists(dest) and os.path.getsize(dest) > 1000:
        return True
    try:
        req = urllib.request.Request(
            url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=20) as r:
            data = r.read()
        if len(data) < 1000:
            return False
        with open(dest, 'wb') as f:
            f.write(data)
        return True
    except Exception:
        return False

class FontManager:
    def __init__(self):
        self._cache: Dict[str, ImageFont.FreeTypeFont] = {}

    def download_all(self) -> int:
        ok = 0
        for alias, info in FONT_CATALOG.items():
            dest = os.path.join(CONFIG.FONT_DIR, info['file'])
            got  = _dl_font(info["url"], dest)
            if not got and info.get("fallback"):
                got = _dl_font(info["fallback"], dest)
            if got:
                ok += 1
                size = os.path.getsize(dest)//1024
                print(f"  ✅ {alias:<22} ({size}KB)")
            else:
                print(f"  ❌ {alias:<22} (failed — system font fallback active)")
        return ok

    def get(self, alias: str, size: int) -> ImageFont.FreeTypeFont:
        key = f"{alias}@{size}"
        if key in self._cache:
            return self._cache[key]
        font = self._load(alias, size)
        self._cache[key] = font
        return font

    def _load(self, alias: str, size: int) -> ImageFont.FreeTypeFont:
        candidates = []
        if alias in FONT_CATALOG:
            candidates.append(
                os.path.join(CONFIG.FONT_DIR, FONT_CATALOG[alias]['file'])
            )
        # Bengali fallbacks
        candidates += [
            os.path.join(CONFIG.FONT_DIR, "bengali_regular.ttf"),
            os.path.join(CONFIG.FONT_DIR, "bengali_bold.ttf"),
        ]
        for p in candidates:
            if os.path.exists(p):
                try:
                    return ImageFont.truetype(p, size)
                except Exception:
                    continue
        try:
            return ImageFont.load_default(size=size)
        except Exception:
            return ImageFont.load_default()

    def clear_cache(self):
        self._cache.clear()
        print("  🗑️  Font cache cleared")

    def ready_count(self) -> int:
        return sum(
            1 for info in FONT_CATALOG.values()
            if os.path.exists(
                os.path.join(CONFIG.FONT_DIR, info['file'])
            )
        )

    # ── Shortcuts ─────────────────────────────────────────────
    def heading(self, size):    return self.get("bengali_bold",     size)
    def subheading(self, size): return self.get("bengali_semibold", size)
    def body(self, size):       return self.get("bengali_regular",  size)
    def light(self, size):      return self.get("bengali_light",    size)
    def en_bold(self, size):    return self.get("english_bold",     size)
    def en_semi(self, size):    return self.get("english_semibold", size)
    def en_reg(self, size):     return self.get("english_regular",  size)
    def en_light(self, size):   return self.get("english_light",    size)
    def en_italic(self, size):  return self.get("english_italic",   size)

    def get_by_scale(self, alias: str, scale_key: str) -> ImageFont.FreeTypeFont:
        """FONT_SCALE key দিয়ে font লোড করুন"""
        size = FONT_SCALE.get(scale_key, 40)
        return self.get(alias, size)

print("\n🔤 Downloading fonts...")
font_manager = FontManager()
font_ok = font_manager.download_all()
print(f"  📊 {font_ok}/{len(FONT_CATALOG)} fonts ready\n")
print("✅ FontManager ready")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 03 — BRAND KIT SYSTEM
# ══════════════════════════════════════════════════════════════

@dataclass
class BrandKitToken:
    name:           str = "Default"
    primary:        str = "#1E293B"
    secondary:      str = "#334155"
    accent:         str = "#6366F1"
    background:     str = "#0F172A"
    text_primary:   str = "#F8FAFC"
    text_secondary: str = "#94A3B8"
    font_heading:   str = "bengali_bold"
    font_body:      str = "bengali_regular"
    font_accent:    str = "english_italic"

    def to_json(self) -> str:
        return json.dumps(asdict(self), indent=2, ensure_ascii=False)

    @classmethod
    def from_json(cls, s: str) -> 'BrandKitToken':
        """Unknown fields gracefully ignore করে (backward compatible)"""
        data  = json.loads(s)
        valid = {f.name for f in dataclasses.fields(cls)}
        return cls(**{k: v for k, v in data.items() if k in valid})

    def get_auto_text_color(self) -> str:
        return auto_text_color(self.background)

    def preview(self):
        """Terminal-friendly preview"""
        print(f"  🎨 {self.name}")
        print(f"     primary:    {self.primary}")
        print(f"     secondary:  {self.secondary}")
        print(f"     accent:     {self.accent}")
        print(f"     background: {self.background}")
        print(f"     text:       {self.text_primary} / {self.text_secondary}")

BRAND_KITS: Dict[str, BrandKitToken] = {
    "ModernTech": BrandKitToken(
        name="ModernTech", primary="#0F172A", secondary="#1E293B",
        accent="#6366F1", background="#020617",
        text_primary="#F8FAFC", text_secondary="#94A3B8",
    ),
    "VibrantCreative": BrandKitToken(
        name="VibrantCreative", primary="#EC4899", secondary="#8B5CF6",
        accent="#FBBF24", background="#1a0533",
        text_primary="#FFFFFF", text_secondary="#F3E8FF",
    ),
    "ElegantMinimal": BrandKitToken(
        name="ElegantMinimal", primary="#FFFFFF", secondary="#F1F5F9",
        accent="#1E293B", background="#F8FAFC",
        text_primary="#0F172A", text_secondary="#475569",
        font_heading="english_italic", font_body="english_light",
    ),
    "DarkNeon": BrandKitToken(
        name="DarkNeon", primary="#000000", secondary="#0D0D0D",
        accent="#00F0FF", background="#050505",
        text_primary="#FFFFFF", text_secondary="#A0A0A0",
    ),
    "BengaliVibrant": BrandKitToken(
        name="BengaliVibrant", primary="#006A4E", secondary="#F42A41",
        accent="#FFD700", background="#003825",
        text_primary="#FFFFFF", text_secondary="#F0F0F0",
    ),
    "BoldCorporate": BrandKitToken(
        name="BoldCorporate", primary="#1D1D1B", secondary="#2C2C2C",
        accent="#E63946", background="#111111",
        text_primary="#FFFFFF", text_secondary="#CCCCCC",
    ),
    "SunriseWarm": BrandKitToken(
        name="SunriseWarm", primary="#FF6B6B", secondary="#FFA726",
        accent="#FFD700", background="#1A0A00",
        text_primary="#FFFFFF", text_secondary="#FFE0C0",
    ),
    "OceanBreeze": BrandKitToken(
        name="OceanBreeze", primary="#0077B6", secondary="#00B4D8",
        accent="#90E0EF", background="#03045E",
        text_primary="#FFFFFF", text_secondary="#CAF0F8",
    ),
}

class BrandKitManager:
    def __init__(self):
        os.makedirs(CONFIG.BRAND_DIR, exist_ok=True)
        for name, kit in BRAND_KITS.items():
            p = os.path.join(CONFIG.BRAND_DIR, f"{name}.json")
            with open(p, "w", encoding="utf-8") as f:
                f.write(kit.to_json())

    def save(self, kit: BrandKitToken) -> str:
        path = os.path.join(CONFIG.BRAND_DIR, f"{kit.name}.json")
        with open(path, "w", encoding="utf-8") as f:
            f.write(kit.to_json())
        BRAND_KITS[kit.name] = kit
        return path

    def load(self, name: str) -> Optional[BrandKitToken]:
        if name in BRAND_KITS:
            return BRAND_KITS[name]
        p = os.path.join(CONFIG.BRAND_DIR, f"{name}.json")
        if os.path.exists(p):
            with open(p, encoding="utf-8") as f:
                return BrandKitToken.from_json(f.read())
        return None

    def list_kits(self) -> List[str]:
        disk = [p.stem for p in Path(CONFIG.BRAND_DIR).glob("*.json")]
        return sorted(set(list(BRAND_KITS.keys()) + disk))

    def get_or_default(self, name: str) -> BrandKitToken:
        """Kit না পেলে ModernTech return করে"""
        return self.load(name) or BRAND_KITS["ModernTech"]

    def create_custom(
        self, name: str, primary: str, accent: str,
        background: str, **kwargs
    ) -> BrandKitToken:
        """দ্রুত custom kit তৈরি করুন"""
        tp   = auto_text_color(background)
        ts   = blend_colors(tp, background, 0.4)
        kit  = BrandKitToken(
            name=name,
            primary=primary,
            secondary=blend_colors(primary, background, 0.5),
            accent=accent,
            background=background,
            text_primary=tp,
            text_secondary=ts,
            **{k: v for k, v in kwargs.items()
               if k in {f.name for f in dataclasses.fields(BrandKitToken)}}
        )
        self.save(kit)
        return kit

brand_manager = BrandKitManager()
print(f"✅ Brand kits: {len(brand_manager.list_kits())} loaded")
print(f"   Kits: {brand_manager.list_kits()}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 04 — GRAPHIC ENGINE
# _refresh_draw + add_divider_line + সব method fix
# ══════════════════════════════════════════════════════════════

_SHAPE_MAP: Dict[str, ImageShape] = {
    "circle": ImageShape.CIRCLE,   "round": ImageShape.CIRCLE,
    "oval":   ImageShape.CIRCLE,
    "star":   ImageShape.STAR,     "burst": ImageShape.STAR,
    "cross":  ImageShape.STAR,
    "hexagon": ImageShape.HEXAGON, "hex":    ImageShape.HEXAGON,
    "shield":  ImageShape.HEXAGON,
    "heart":  ImageShape.HEART,    "love":   ImageShape.HEART,
    "rounded_square": ImageShape.ROUNDED_SQUARE,
    "rounded":        ImageShape.ROUNDED_SQUARE,
    "square":         ImageShape.ROUNDED_SQUARE,
    "badge":          ImageShape.ROUNDED_SQUARE,
    "pill":           ImageShape.ROUNDED_SQUARE,
    "diamond":  ImageShape.DIAMOND,   "rhombus":  ImageShape.DIAMOND,
    "triangle": ImageShape.TRIANGLE,  "arrow_up": ImageShape.TRIANGLE,
    "blob":     ImageShape.CIRCLE,
}

class GraphicEngine:
    """Core canvas engine — সব ডিজাইনের ভিত্তি"""

    def __init__(self,
                 width:    int = CONFIG.DEFAULT_WIDTH,
                 height:   int = CONFIG.DEFAULT_HEIGHT,
                 bg_color: str = "#FFFFFF"):
        self.width  = int(clamp(width,  100, CONFIG.MAX_CANVAS_SIZE))
        self.height = int(clamp(height, 100, CONFIG.MAX_CANVAS_SIZE))
        self.canvas = Image.new(
            "RGBA", (self.width, self.height),
            hex_with_alpha(bg_color, 1.0)
        )
        self.draw  = ImageDraw.Draw(self.canvas)
        self._kit: Optional[BrandKitToken] = None

    def __enter__(self):  return self
    def __exit__(self, *a): pass

    def _refresh_draw(self):
        """canvas replace হলে draw object refresh — সব method এটা call করে"""
        self.draw = ImageDraw.Draw(self.canvas)

    # ── Background ────────────────────────────────────────────
    def create_solid_background(self, color: str) -> 'GraphicEngine':
        self.canvas.paste(
            Image.new("RGBA", (self.width, self.height),
                      hex_with_alpha(color, 1.0))
        )
        self._refresh_draw()
        return self

    def create_gradient_background(
        self,
        color1:    str,
        color2:    str,
        direction: GradientDirection = GradientDirection.VERTICAL,
        color3:    Optional[str]     = None,
    ) -> 'GraphicEngine':
        W, H     = self.width, self.height
        r1,g1,b1 = hex_to_rgb(color1)
        r2,g2,b2 = hex_to_rgb(color2)
        arr = np.zeros((H, W, 4), dtype=np.uint8)

        if direction == GradientDirection.RADIAL:
            cx, cy = W/2, H/2
            maxd   = math.sqrt(cx**2 + cy**2)
            ys, xs = np.mgrid[0:H, 0:W]
            d      = np.sqrt((xs-cx)**2 + (ys-cy)**2)
            t      = np.clip(d / maxd, 0, 1)
        elif direction == GradientDirection.HORIZONTAL:
            t = np.tile(np.linspace(0,1,W), (H,1))
        elif direction == GradientDirection.DIAGONAL:
            row_t = np.linspace(0,1,H)[:,None]
            col_t = np.linspace(0,1,W)[None,:]
            t     = (row_t + col_t) / 2
        else:  # VERTICAL
            t = np.tile(np.linspace(0,1,H)[:,None], (1,W))

        if color3:
            r3,g3,b3 = hex_to_rgb(color3)
            t2 = np.clip(t*2,   0, 1)
            t1 = np.clip(t*2-1, 0, 1)
            arr[:,:,0] = np.where(t<0.5, r1+(r2-r1)*t2, r2+(r3-r2)*t1).astype(np.uint8)
            arr[:,:,1] = np.where(t<0.5, g1+(g2-g1)*t2, g2+(g3-g1)*t1).astype(np.uint8)
            arr[:,:,2] = np.where(t<0.5, b1+(b2-b1)*t2, b2+(b3-b1)*t1).astype(np.uint8)
        else:
            arr[:,:,0] = (r1 + (r2-r1)*t).astype(np.uint8)
            arr[:,:,1] = (g1 + (g2-g1)*t).astype(np.uint8)
            arr[:,:,2] = (b1 + (b2-b1)*t).astype(np.uint8)

        arr[:,:,3] = 255
        self.canvas = Image.fromarray(arr, "RGBA")
        self._refresh_draw()
        return self

    def set_background_image(
        self, image_bytes: bytes,
        blur: float = 0, brightness: float = 1.0,
        overlay_color: Optional[str] = None,
        overlay_alpha: float = 0.5,
    ) -> 'GraphicEngine':
        img   = Image.open(io.BytesIO(image_bytes)).convert("RGBA")
        img_r = img.width / img.height
        can_r = self.width / self.height
        if img_r > can_r:
            new_h = self.height
            new_w = int(img.width * self.height / img.height)
        else:
            new_w = self.width
            new_h = int(img.height * self.width / img.width)
        img  = img.resize((new_w, new_h), Image.Resampling.LANCZOS)
        left = (new_w - self.width)  // 2
        top  = (new_h - self.height) // 2
        img  = img.crop((left, top, left+self.width, top+self.height))
        if blur > 0:
            img = img.filter(ImageFilter.GaussianBlur(blur))
        if brightness != 1.0:
            img = ImageEnhance.Brightness(img).enhance(brightness)
        self.canvas = img.copy()
        self._refresh_draw()
        if overlay_color:
            self.add_color_overlay(overlay_color, overlay_alpha)
        return self

    def add_color_overlay(
        self, color: str, alpha: float = 0.5
    ) -> 'GraphicEngine':
        if alpha > 1.0:
            alpha = alpha / 100.0
        alpha   = clamp(float(alpha), 0, 1)
        overlay = Image.new(
            "RGBA", (self.width, self.height),
            hex_with_alpha(color, alpha)
        )
        self.canvas = Image.alpha_composite(self.canvas, overlay)
        self._refresh_draw()
        return self

    # ── Shapes ────────────────────────────────────────────────
    def add_rectangle(
        self,
        x1: int, y1: int, x2: int, y2: int,
        fill:          Optional[str] = None,
        outline:       Optional[str] = None,
        outline_width: int   = 2,
        radius:        int   = 0,
        alpha:         float = 1.0,
        opacity:       Optional[float] = None,
    ) -> 'GraphicEngine':
        if opacity is not None:
            alpha = opacity/100.0 if opacity > 1.0 else float(opacity)
        alpha = clamp(float(alpha), 0, 1)
        layer = Image.new("RGBA", (self.width, self.height), (0,0,0,0))
        d     = ImageDraw.Draw(layer)
        fc    = hex_with_alpha(fill,    alpha) if fill    else None
        oc    = hex_with_alpha(outline, alpha) if outline else None
        if radius > 0:
            d.rounded_rectangle(
                [x1,y1,x2,y2], radius=radius,
                fill=fc, outline=oc, width=outline_width
            )
        else:
            d.rectangle(
                [x1,y1,x2,y2],
                fill=fc, outline=oc, width=outline_width
            )
        self.canvas = Image.alpha_composite(self.canvas, layer)
        self._refresh_draw()
        return self

    def add_circle(
        self, cx: int, cy: int, r: int,
        fill: Optional[str] = None,
        outline: Optional[str] = None,
        outline_width: int = 2,
        alpha: float = 1.0,
        opacity: Optional[float] = None,
    ) -> 'GraphicEngine':
        if opacity is not None:
            alpha = opacity/100.0 if opacity > 1.0 else float(opacity)
        return self.add_rectangle(
            cx-r, cy-r, cx+r, cy+r,
            fill=fill, outline=outline,
            outline_width=outline_width,
            radius=r, alpha=clamp(float(alpha), 0, 1)
        )

    def add_line(
        self, x1, y1, x2, y2,
        color: str = "#FFFFFF",
        width: int = 2,
        alpha: float = 1.0,
        opacity: Optional[float] = None,
    ) -> 'GraphicEngine':
        if opacity is not None:
            alpha = opacity/100.0 if opacity > 1.0 else float(opacity)
        alpha = clamp(float(alpha), 0, 1)
        layer = Image.new("RGBA", (self.width, self.height), (0,0,0,0))
        ImageDraw.Draw(layer).line(
            [(x1,y1),(x2,y2)],
            fill=hex_with_alpha(color, alpha),
            width=width
        )
        self.canvas = Image.alpha_composite(self.canvas, layer)
        self._refresh_draw()
        return self

    def add_polygon(
        self, points: List[Tuple[int,int]],
        fill: Optional[str] = None,
        outline: Optional[str] = None,
        outline_width: int = 2,
        alpha: float = 1.0,
    ) -> 'GraphicEngine':
        layer = Image.new("RGBA", (self.width, self.height), (0,0,0,0))
        d     = ImageDraw.Draw(layer)
        fc    = hex_with_alpha(fill,    alpha) if fill    else None
        oc    = hex_with_alpha(outline, alpha) if outline else None
        d.polygon(points, fill=fc, outline=oc)
        self.canvas = Image.alpha_composite(self.canvas, layer)
        self._refresh_draw()
        return self

    # ── Decorators ────────────────────────────────────────────
    def add_corner_decoration(
        self, color: str, size: int,
        corner: str  = "top-right",
        shape:  str  = "triangle",
        opacity_pct: float = 100,
    ) -> 'GraphicEngine':
        alpha = clamp(opacity_pct / 100, 0.0, 1.0)
        W, H  = self.width, self.height
        layer = Image.new("RGBA", (W, H), (0,0,0,0))
        d     = ImageDraw.Draw(layer)
        c     = hex_with_alpha(color, alpha)
        s     = size
        if shape == "triangle":
            pts_map = {
                "top-right":    [(W-s,0),(W,0),(W,s)],
                "top-left":     [(0,0),(s,0),(0,s)],
                "bottom-right": [(W,H-s),(W,H),(W-s,H)],
                "bottom-left":  [(0,H-s),(s,H),(0,H)],
            }
            d.polygon(pts_map.get(corner, [(W-s,0),(W,0),(W,s)]), fill=c)
        elif shape == "square":
            rects = {
                "top-right":    (W-s, 0, W,   s),
                "top-left":     (0,   0, s,   s),
                "bottom-right": (W-s, H-s, W, H),
                "bottom-left":  (0,   H-s, s, H),
            }
            x1,y1,x2,y2 = rects.get(corner, (W-s,0,W,s))
            d.rectangle([x1,y1,x2,y2], fill=c)
        elif shape == "circle":
            off = {
                "top-right":    (W-s, -s, W+s,  s),
                "top-left":     (-s,  -s,  s,   s),
                "bottom-right": (W-s, H-s, W+s, H+s),
                "bottom-left":  (-s,  H-s,  s,  H+s),
            }
            d.ellipse(off.get(corner, (W-s,-s,W+s,s)), fill=c)
        elif shape == "arc":
            arcs = {
                "top-right":    (W-s*2, -s, W+s, s,       180, 270),
                "top-left":     (-s,    -s,  s,  s,       270, 360),
                "bottom-right": (W-s, H-s, W+s, H+s,      90, 180),
                "bottom-left":  (-s,  H-s,  s,  H+s,       0,  90),
            }
            x1,y1,x2,y2,st,en = arcs.get(corner, (W-s*2,-s,W+s,s,180,270))
            r_,g_,b_ = hex_to_rgb(color)
            d.pieslice([x1,y1,x2,y2], start=st, end=en,
                       fill=(r_,g_,b_,int(255*alpha)))
        self.canvas = Image.alpha_composite(self.canvas, layer)
        self._refresh_draw()
        return self

    def add_decorative_corners(
        self, color: str, size: int = 150,
        shape: str = "triangle",
        corners: Optional[List[str]] = None,
        opacity_pct: float = 100,
    ) -> 'GraphicEngine':
        all_c = ["top-left","top-right","bottom-left","bottom-right"]
        for corner in (corners or all_c):
            self.add_corner_decoration(color, size, corner,
                                       shape, opacity_pct)
        return self

    def add_badge(
        self, text: str, x: int, y: int,
        bg_color:   str = "#E63946",
        text_color: str = "#FFFFFF",
        font_alias: str = "bengali_bold",
        font_size:  int = 36,
        padding_x:  int = 24,
        padding_y:  int = 12,
        radius:     int = 10,
        alpha:      float = 1.0,
    ) -> 'GraphicEngine':
        font = font_manager.get(font_alias, font_size)
        tmp  = ImageDraw.Draw(Image.new("RGBA", (1,1)))
        bb   = tmp.textbbox((0,0), text, font=font)
        tw   = bb[2] - bb[0]
        th   = bb[3] - bb[1]
        bw   = tw + padding_x * 2
        bh   = th + padding_y * 2
        self.add_rectangle(x, y, x+bw, y+bh,
                           fill=bg_color, radius=radius, alpha=alpha)
        self.add_text(text, x=x+padding_x, y=y+padding_y,
                      font_alias=font_alias, font_size=font_size,
                      color=text_color, alpha=alpha)
        return self

    def add_divider(
        self, y: int,
        color:     str   = "#FFFFFF",
        alpha:     float = 0.3,
        thickness: int   = 2,
        style:     str   = "solid",
        margin:    int   = 60,
    ) -> 'GraphicEngine':
        if alpha > 1.0:
            alpha = alpha / 100.0
        alpha = clamp(float(alpha), 0, 1)
        x1, x2 = margin, self.width - margin
        if style == "dashed":
            cx = x1
            while cx < x2:
                self.add_line(cx, y, min(cx+30, x2), y,
                              color=color, width=thickness, alpha=alpha)
                cx += 45
        elif style == "dotted":
            cx = x1
            while cx < x2:
                self.add_circle(cx, y, thickness,
                                fill=color, alpha=alpha)
                cx += 12
        elif style == "double":
            self.add_line(x1, y-5, x2, y-5,
                          color=color, width=thickness, alpha=alpha)
            self.add_line(x1, y+5, x2, y+5,
                          color=color, width=thickness, alpha=alpha)
        else:  # solid
            self.add_line(x1, y, x2, y,
                          color=color, width=thickness, alpha=alpha)
        return self

    # ✅ Alias — Cell 12 TemplateEngine যেটা call করে
    def add_divider_line(
        self, y: int,
        color:     str   = "#FFFFFF",
        alpha:     float = 0.3,
        thickness: int   = 2,
        style:     str   = "solid",
        margin:    int   = 60,
    ) -> 'GraphicEngine':
        return self.add_divider(
            y, color=color, alpha=alpha,
            thickness=thickness, style=style, margin=margin
        )

    def add_accent_line(
        self, y: int,
        color:     str   = "#6366F1",
        width_pct: float = 0.3,
        thickness: int   = 4,
        align:     str   = "center",
        alpha:     float = 1.0,
    ) -> 'GraphicEngine':
        lw = int(self.width * clamp(width_pct, 0.05, 1.0))
        if align == "left":
            x1 = 60
        elif align == "right":
            x1 = self.width - 60 - lw
        else:
            x1 = (self.width - lw) // 2
        return self.add_line(x1, y, x1+lw, y,
                             color=color, width=thickness, alpha=alpha)

    # ── Text ──────────────────────────────────────────────────
    def _wrap_text(
        self, text: str,
        font: ImageFont.FreeTypeFont,
        max_width: int
    ) -> List[str]:
        max_width = safe_int(max_width, self.width - 80)
        if max_width <= 0:
            max_width = self.width - 80
        tmp_d = ImageDraw.Draw(Image.new("RGBA", (1,1)))
        words = text.split()
        lines: List[str] = []
        line  = ""
        for w in words:
            test = (line + " " + w).strip()
            if tmp_d.textlength(test, font=font) <= max_width:
                line = test
            else:
                if line:
                    lines.append(line)
                line = w
        if line:
            lines.append(line)
        return lines or [text]

    def add_text(
        self,
        text:          str,
        x:             Union[int, str] = 'center',
        y:             int             = 100,
        font_alias:    str             = "bengali_bold",
        font_size:     int             = 60,
        color:         str             = "#FFFFFF",
        alpha:         float           = 1.0,
        opacity:       Optional[float] = None,
        max_width:     Optional[int]   = None,
        line_spacing:  int             = 12,
        align:         str             = "center",
        shadow:        bool            = False,
        shadow_color:  str             = "#000000",
        shadow_offset: Tuple           = (3, 3),
        shadow_alpha:  float           = 0.5,
        font_size_key: Optional[str]   = None,
    ) -> Tuple[int, int, int, int]:
        """Text আঁকে — returns (x1,y1,x2,y2) bounding box"""
        if opacity is not None:
            alpha = opacity/100.0 if opacity > 1.0 else float(opacity)
        alpha = clamp(float(alpha), 0, 1)
        if font_size_key and font_size_key in FONT_SCALE:
            font_size = FONT_SCALE[font_size_key]
        font   = font_manager.get(font_alias, font_size)
        lines  = self._wrap_text(text, font, max_width or self.width - 80)
        fill_c = hex_with_alpha(color, alpha)
        layer  = Image.new("RGBA", (self.width, self.height), (0,0,0,0))
        d      = ImageDraw.Draw(layer)
        line_h = font_size + line_spacing
        cur_y  = y
        x1_all = self.width; x2_all = 0
        y1_all = y;          y2_all = y

        for line in lines:
            bb = d.textbbox((0,0), line, font=font)
            lw = bb[2] - bb[0]
            if x == 'center':
                lx = (self.width - lw) // 2
            elif x == 'right':
                lx = self.width - lw - 40
            elif x == 'left':
                lx = 40
            else:
                lx = safe_int(x, (self.width - lw) // 2)

            if shadow:
                sx, sy = (shadow_offset
                          if isinstance(shadow_offset, tuple)
                          else (shadow_offset, shadow_offset))
                sc = hex_with_alpha(
                    shadow_color,
                    clamp(shadow_alpha * alpha, 0, 1)
                )
                for ox, oy in [(-1,-1),(1,-1),(-1,1),(1,1),(sx,sy)]:
                    d.text((lx+ox, cur_y+oy), line, font=font, fill=sc)

            d.text((lx, cur_y), line, font=font, fill=fill_c)
            x1_all = min(x1_all, lx)
            x2_all = max(x2_all, lx + lw)
            y2_all = cur_y + line_h
            cur_y += line_h

        self.canvas = Image.alpha_composite(self.canvas, layer)
        self._refresh_draw()
        return (x1_all, y1_all, x2_all, y2_all)

    # ── Image paste ───────────────────────────────────────────
    def _apply_shape_mask(
        self, img: Image.Image,
        shape: Union[ImageShape, str]
    ) -> Image.Image:
        if isinstance(shape, str):
            shape = _SHAPE_MAP.get(
                shape.lower().strip(),
                ImageShape.ROUNDED_SQUARE
            )
        w, h = img.size
        mask = Image.new("L", (w, h), 0)
        d    = ImageDraw.Draw(mask)

        if shape == ImageShape.CIRCLE:
            d.ellipse([0,0,w,h], fill=255)
        elif shape == ImageShape.ROUNDED_SQUARE:
            r = min(w,h) // 5
            d.rounded_rectangle([0,0,w,h], radius=r, fill=255)
        elif shape == ImageShape.STAR:
            cx, cy = w//2, h//2
            ro, ri = min(w,h)//2, min(w,h)//4
            pts = []
            for i in range(10):
                ang = math.pi/2 + 2*math.pi*i/10
                r_  = ro if i%2==0 else ri
                pts.append((cx + r_*math.cos(ang),
                             cy - r_*math.sin(ang)))
            d.polygon(pts, fill=255)
        elif shape == ImageShape.HEXAGON:
            cx, cy = w//2, h//2
            r_     = min(w,h)//2
            pts    = [
                (cx + r_*math.cos(math.pi/3*i),
                 cy + r_*math.sin(math.pi/3*i))
                for i in range(6)
            ]
            d.polygon(pts, fill=255)
        elif shape == ImageShape.HEART:
            cx, cy = w//2, h//2
            sc     = min(w,h)//16
            pts    = []
            for t_deg in range(0, 360, 3):
                t  = math.radians(t_deg)
                hx = 16 * math.sin(t)**3
                hy = -(13*math.cos(t) - 5*math.cos(2*t)
                        - 2*math.cos(3*t) - math.cos(4*t))
                pts.append((cx + hx*sc, cy + hy*sc))
            d.polygon(pts, fill=255)
        elif shape == ImageShape.DIAMOND:
            pts = [(w//2,0),(w,h//2),(w//2,h),(0,h//2)]
            d.polygon(pts, fill=255)
        elif shape == ImageShape.TRIANGLE:
            pts = [(w//2,0),(w,h),(0,h)]
            d.polygon(pts, fill=255)
        else:
            mask = Image.new("L", (w, h), 255)

        result = Image.new("RGBA", (w, h), (0,0,0,0))
        result.paste(img, (0,0), mask=mask)
        return result

    def paste_image(
        self,
        image_bytes:   bytes,
        x:             Union[int,str] = 'center',
        y:             Union[int,str] = 'center',
        width:         Optional[int]  = None,
        height:        Optional[int]  = None,
        shape:         Union[ImageShape, str] = ImageShape.CIRCLE,
        outline_color: Optional[str]  = None,
        outline_width: int            = 0,
        remove_bg:     bool           = False,
    ) -> 'GraphicEngine':
        img = Image.open(io.BytesIO(image_bytes)).convert("RGBA")

        if remove_bg:
            img = remove_background(img)

        # Resize
        if width and height:
            img = img.resize((width, height), Image.Resampling.LANCZOS)
        elif width:
            r   = width / img.width
            img = img.resize(
                (width, int(img.height*r)), Image.Resampling.LANCZOS)
        elif height:
            r   = height / img.height
            img = img.resize(
                (int(img.width*r), height), Image.Resampling.LANCZOS)

        img = self._apply_shape_mask(img, shape)

        # Outline
        if outline_color and outline_width > 0:
            ow  = outline_width
            out = Image.new(
                "RGBA",
                (img.width + ow*2, img.height + ow*2),
                (0,0,0,0)
            )
            od      = ImageDraw.Draw(out)
            r_,g_,b_ = hex_to_rgb(outline_color)
            od.ellipse([0, 0, out.width, out.height],
                       fill=(r_,g_,b_,255))
            out.paste(img, (ow, ow), img)
            img = out

        iw, ih = img.size
        if x == 'center':  px = (self.width  - iw) // 2
        elif x == 'right':  px = self.width - iw - 20
        elif x == 'left':   px = 20
        else:               px = safe_int(x, (self.width-iw)//2)

        if y == 'center':  py = (self.height - ih) // 2
        elif y == 'bottom': py = self.height - ih
        elif y == 'top':    py = 0
        else:               py = safe_int(y, (self.height-ih)//2)

        self.canvas.paste(img, (px, py), img)
        self._refresh_draw()
        return self

    # ── QR Code ───────────────────────────────────────────────
    def add_qr_code(
        self,
        data:       str,
        x:          int = 800,
        y:          int = 800,
        size:       int = 200,
        fill_color: str = "#000000",
        back_color: str = "#FFFFFF",
    ) -> 'GraphicEngine':
        qr = qrcode.QRCode(
            box_size=6, border=2,
            error_correction=qrcode.constants.ERROR_CORRECT_H
        )
        qr.add_data(data)
        qr.make(fit=True)
        qr_img = qr.make_image(
            fill_color=fill_color, back_color=back_color
        ).convert("RGBA")
        qr_img = qr_img.resize((size, size), Image.Resampling.LANCZOS)
        self.canvas.paste(qr_img, (x, y), qr_img)
        self._refresh_draw()
        return self

    # ── Save / Show ───────────────────────────────────────────
    def get_rgb(self) -> Image.Image:
        return self.canvas.convert("RGB")

    def show(self, max_width: int = 600) -> None:
        img = self.get_rgb()
        if img.width > max_width:
            r   = max_width / img.width
            img = img.resize(
                (max_width, int(img.height*r)),
                Image.Resampling.LANCZOS
            )
        display(img)

    def copy(self) -> 'GraphicEngine':
        """এই engine-এর exact copy তৈরি করুন"""
        new_eng        = GraphicEngine(self.width, self.height)
        new_eng.canvas = self.canvas.copy()
        new_eng._refresh_draw()
        return new_eng

    def save(
        self,
        filename:  str,
        fmt:       str = "PNG",
        quality:   int = CONFIG.JPEG_QUALITY,
        directory: str = CONFIG.OUTPUT_DIR,
    ) -> str:
        os.makedirs(directory, exist_ok=True)
        path = os.path.join(directory, filename)
        rgb  = self.get_rgb()
        if fmt == "JPEG":
            rgb.save(path, "JPEG", quality=quality, optimize=True)
        elif fmt == "WEBP":
            rgb.save(path, "WEBP", quality=quality)
        else:
            self.canvas.save(path, "PNG")
        return path


# ── Quick Test ────────────────────────────────────────────────
print("\n🧪 GraphicEngine Test:")
with GraphicEngine(800, 300) as eng:
    eng.create_gradient_background(
        "#0F0C29", "#302B63", GradientDirection.DIAGONAL)
    eng.add_rectangle(40, 40, 760, 260,
                      fill="#FFFFFF", alpha=0.05, radius=20)
    eng.add_divider(150, color="#6366F1", thickness=2, alpha=0.6)
    eng.add_text("GraphicEngine ✅ সব method ready",
                 x='center', y=80,
                 font_alias="bengali_bold",
                 font_size=48, color="#FFFFFF")
    eng.add_accent_line(200, color="#00F0FF",
                        width_pct=0.25, align="center")
    eng.show()
    eng.save("engine_test.png")
print("✅ GraphicEngine ready — _refresh_draw ✅  add_divider_line ✅")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 05 — VISUAL AESTHETICS ENGINE
# Glassmorphism, Texture, Shadow, Bokeh, Vignette, Color Grade
# ══════════════════════════════════════════════════════════════

class VisualAestheticsEngine:
    """
    পেশাদার Visual Effects ইঞ্জিন

    Effects:
    ─────────
    ✅ Glassmorphism (frosted glass UI)
    ✅ Texture overlays (paper, grain, geometric, dots, lines)
    ✅ Advanced shadow & glow
    ✅ Vignette effect
    ✅ Noise overlay
    ✅ Grid & dot patterns
    ✅ Bokeh light effects
    ✅ Color grading (cinematic LUT)
    """

    # ── Glassmorphism ─────────────────────────────────────────
    @staticmethod
    def add_glassmorphism(
        engine: GraphicEngine,
        x1: int, y1: int, x2: int, y2: int,
        blur_radius:  float = 12.0,
        fill_opacity: float = 0.15,
        border_color: str   = "#FFFFFF",
        border_opacity: float = 0.25,
        border_width: int   = 1,
        radius:       int   = 20,
        tint_color:   str   = "#FFFFFF",
    ) -> GraphicEngine:
        rw = max(1, x2 - x1)
        rh = max(1, y2 - y1)

        # Crop + blur region
        region = engine.canvas.crop((x1, y1, x2, y2)).convert("RGBA")
        blurred = region.filter(ImageFilter.GaussianBlur(blur_radius))

        # Tint overlay
        tint = Image.new("RGBA", (rw, rh),
                         hex_with_alpha(tint_color, fill_opacity))
        glass = Image.alpha_composite(blurred, tint)

        # Shape mask
        mask = Image.new("L", (rw, rh), 0)
        ImageDraw.Draw(mask).rounded_rectangle(
            [0, 0, rw, rh], radius=radius, fill=255
        )
        alpha_ch = Image.new("L", (rw, rh), 255)
        glass.putalpha(alpha_ch)

        glass_masked = Image.new("RGBA", (rw, rh), (0,0,0,0))
        glass_masked.paste(glass, (0,0), mask=mask)
        engine.canvas.paste(glass_masked, (x1, y1), glass_masked)

        # Border
        if border_opacity > 0 and border_width > 0:
            engine.add_rectangle(
                x1, y1, x2, y2,
                outline=border_color,
                outline_width=border_width,
                radius=radius,
                alpha=border_opacity
            )

        engine._refresh_draw()
        return engine

    # ── Texture ───────────────────────────────────────────────
    @staticmethod
    def apply_texture(
        engine:   GraphicEngine,
        texture:  TextureType = TextureType.GRAIN,
        strength: float       = 0.15,
        scale:    int         = 3,
    ) -> GraphicEngine:
        W, H  = engine.width, engine.height
        rng   = np.random.default_rng(42)
        layer = np.zeros((H, W, 4), dtype=np.uint8)

        if texture == TextureType.GRAIN:
            noise = rng.integers(0, 255, (H, W), dtype=np.uint8)
            layer[:,:,0] = layer[:,:,1] = layer[:,:,2] = noise
            layer[:,:,3] = int(255 * clamp(strength, 0, 1))

        elif texture == TextureType.PAPER:
            base  = rng.integers(200, 240, (H, W), dtype=np.uint8)
            noise = rng.integers(-20, 20, (H, W)).astype(np.int16)
            paper = np.clip(base.astype(np.int16) + noise, 180, 255).astype(np.uint8)
            layer[:,:,0] = layer[:,:,1] = layer[:,:,2] = paper
            layer[:,:,3] = int(255 * clamp(strength * 0.8, 0, 1))

        elif texture == TextureType.LINES:
            for y in range(0, H, max(1, scale*4)):
                layer[y:y+1, :, 0] = 255
                layer[y:y+1, :, 1] = 255
                layer[y:y+1, :, 2] = 255
                layer[y:y+1, :, 3] = int(255 * clamp(strength, 0, 1))

        elif texture == TextureType.DOTS:
            sp = max(1, scale * 6)
            for yd in range(0, H, sp):
                for xd in range(0, W, sp):
                    y1e = min(yd+2, H)
                    x1e = min(xd+2, W)
                    layer[yd:y1e, xd:x1e, 0] = 255
                    layer[yd:y1e, xd:x1e, 1] = 255
                    layer[yd:y1e, xd:x1e, 2] = 255
                    layer[yd:y1e, xd:x1e, 3] = int(255 * clamp(strength, 0, 1))

        elif texture == TextureType.GEOMETRIC:
            sp = max(1, scale * 20)
            for yd in range(0, H, sp):
                layer[yd:yd+1, :, :3] = 200
                layer[yd:yd+1, :,  3] = int(255 * clamp(strength * 0.5, 0, 1))
            for xd in range(0, W, sp):
                layer[:, xd:xd+1, :3] = 200
                layer[:, xd:xd+1,  3] = int(255 * clamp(strength * 0.5, 0, 1))

        if texture != TextureType.NONE:
            tex_img = Image.fromarray(layer, "RGBA")
            engine.canvas = Image.alpha_composite(engine.canvas, tex_img)
            engine._refresh_draw()
        return engine

    # ── Shadow & Glow ─────────────────────────────────────────
    @staticmethod
    def add_drop_shadow(
        engine: GraphicEngine,
        x1: int, y1: int, x2: int, y2: int,
        shadow_color:  str   = "#000000",
        shadow_opacity: float = 0.5,
        blur_radius:   float = 15.0,
        offset_x:      int   = 8,
        offset_y:      int   = 8,
        radius:        int   = 0,
    ) -> GraphicEngine:
        sw, sh = engine.width, engine.height
        shadow = Image.new("RGBA", (sw, sh), (0,0,0,0))
        d      = ImageDraw.Draw(shadow)
        r_,g_,b_ = hex_to_rgb(shadow_color)
        sc     = (r_, g_, b_, int(255 * clamp(shadow_opacity, 0, 1)))
        if radius > 0:
            d.rounded_rectangle(
                [x1+offset_x, y1+offset_y, x2+offset_x, y2+offset_y],
                radius=radius, fill=sc
            )
        else:
            d.rectangle(
                [x1+offset_x, y1+offset_y, x2+offset_x, y2+offset_y],
                fill=sc
            )
        shadow = shadow.filter(ImageFilter.GaussianBlur(blur_radius))
        engine.canvas = Image.alpha_composite(engine.canvas, shadow)
        engine._refresh_draw()
        return engine

    @staticmethod
    def add_glow(
        engine: GraphicEngine,
        cx: int, cy: int, radius: int,
        color:   str   = "#6366F1",
        opacity: float = 0.4,
        layers:  int   = 5,
    ) -> GraphicEngine:
        for i in range(layers, 0, -1):
            r_   = int(radius * i / layers)
            alph = opacity * (1 - (i-1)/layers) * 0.6
            engine.add_circle(cx, cy, r_, fill=color,
                              alpha=clamp(alph, 0, 1))
        return engine

    @staticmethod
    def add_text_glow(
        engine: GraphicEngine,
        text: str, x: Union[int,str], y: int,
        font_alias: str = "bengali_bold",
        font_size:  int = 60,
        text_color: str = "#FFFFFF",
        glow_color: str = "#6366F1",
        glow_layers: int = 4,
        glow_spread: int = 6,
    ) -> GraphicEngine:
        for i in range(glow_layers, 0, -1):
            off   = glow_spread * i // glow_layers
            alpha = 0.15 * (1 - (i-1)/glow_layers)
            for ox, oy in [
                (-off,-off),(off,-off),(-off,off),(off,off),
                (0,-off),(0,off),(-off,0),(off,0)
            ]:
                ix = (x if isinstance(x, str)
                      else safe_int(x, engine.width//2) + ox)
                engine.add_text(
                    text, x=ix, y=y+oy,
                    font_alias=font_alias, font_size=font_size,
                    color=glow_color, alpha=clamp(alpha, 0, 1)
                )
        engine.add_text(text, x=x, y=y,
                        font_alias=font_alias, font_size=font_size,
                        color=text_color, alpha=1.0)
        return engine

    # ── Vignette ──────────────────────────────────────────────
    @staticmethod
    def add_vignette(
        engine:   GraphicEngine,
        strength: float = 0.6,
        color:    str   = "#000000",
        feather:  float = 0.7,
    ) -> GraphicEngine:
        W, H  = engine.width, engine.height
        layer = np.zeros((H, W, 4), dtype=np.uint8)
        cx, cy = W/2, H/2
        maxd   = math.sqrt(cx**2 + cy**2)
        ys, xs = np.mgrid[0:H, 0:W]
        dist   = np.sqrt((xs-cx)**2 + (ys-cy)**2)
        t      = np.clip(dist / maxd, 0, 1)
        t      = np.clip((t - (1-feather)) / feather, 0, 1)
        alpha  = (t**2 * strength * 255).astype(np.uint8)
        r_,g_,b_ = hex_to_rgb(color)
        layer[:,:,0] = r_
        layer[:,:,1] = g_
        layer[:,:,2] = b_
        layer[:,:,3] = alpha
        vig_img = Image.fromarray(layer, "RGBA")
        engine.canvas = Image.alpha_composite(engine.canvas, vig_img)
        engine._refresh_draw()
        return engine

    # ── Bokeh ─────────────────────────────────────────────────
    @staticmethod
    def add_bokeh_lights(
        engine:       GraphicEngine,
        count:        int        = 15,
        colors:       List[str]  = None,
        size_range:   Tuple      = (30, 120),
        opacity_range: Tuple     = (0.03, 0.2),
        seed:         int        = 42,
    ) -> GraphicEngine:
        colors  = colors or ["#6366F1","#EC4899","#F59E0B",
                             "#10B981","#00F0FF"]
        rng     = np.random.default_rng(seed)
        W, H    = engine.width, engine.height
        layer   = Image.new("RGBA", (W, H), (0,0,0,0))
        d       = ImageDraw.Draw(layer)

        for _ in range(count):
            r_    = int(rng.integers(size_range[0], size_range[1]))
            cx    = int(rng.integers(0, W))
            cy    = int(rng.integers(0, H))
            alpha = float(rng.uniform(opacity_range[0], opacity_range[1]))
            color = colors[int(rng.integers(0, len(colors)))]
            rc, gc, bc = hex_to_rgb(color)
            d.ellipse(
                [cx-r_, cy-r_, cx+r_, cy+r_],
                fill=(rc, gc, bc, int(255*clamp(alpha,0,1)))
            )

        blurred = layer.filter(ImageFilter.GaussianBlur(
            max(1, size_range[1]//6)))
        engine.canvas = Image.alpha_composite(engine.canvas, blurred)
        engine._refresh_draw()  # ✅ Bug fix — _refresh_draw call
        return engine

    # ── Color Grade ───────────────────────────────────────────
    @staticmethod
    def apply_color_grade(
        engine:     GraphicEngine,
        preset:     str   = "cinematic",
        saturation: float = 1.1,
        contrast:   float = 1.05,
        brightness: float = 1.0,
        warmth:     float = 0.0,
    ) -> GraphicEngine:
        img = engine.canvas.convert("RGB")

        presets = {
            "cinematic": {"saturation":0.9, "contrast":1.1, "brightness":0.95, "warmth":0.05},
            "vibrant":   {"saturation":1.3, "contrast":1.1, "brightness":1.05, "warmth":0.0},
            "muted":     {"saturation":0.75,"contrast":0.95,"brightness":1.0,  "warmth":0.0},
            "warm":      {"saturation":1.0, "contrast":1.05,"brightness":1.0,  "warmth":0.15},
            "cool":      {"saturation":1.0, "contrast":1.05,"brightness":1.0,  "warmth":-0.1},
            "bw":        {"saturation":0.0, "contrast":1.2, "brightness":1.0,  "warmth":0.0},
        }
        if preset in presets:
            p = presets[preset]
            saturation = p["saturation"]
            contrast   = p["contrast"]
            brightness = p["brightness"]
            warmth     = p["warmth"]

        img = ImageEnhance.Color(img).enhance(saturation)
        img = ImageEnhance.Contrast(img).enhance(contrast)
        img = ImageEnhance.Brightness(img).enhance(brightness)

        if abs(warmth) > 0.01:
            arr = np.array(img).astype(np.float32)
            if warmth > 0:
                arr[:,:,0] = np.clip(arr[:,:,0]*(1+warmth*0.5), 0, 255)
                arr[:,:,2] = np.clip(arr[:,:,2]*(1-warmth*0.3), 0, 255)
            else:
                arr[:,:,2] = np.clip(arr[:,:,2]*(1+abs(warmth)*0.5), 0, 255)
                arr[:,:,0] = np.clip(arr[:,:,0]*(1-abs(warmth)*0.3), 0, 255)
            img = Image.fromarray(arr.astype(np.uint8), "RGB")

        engine.canvas = img.convert("RGBA")
        engine._refresh_draw()
        return engine

    # ── Noise Overlay ─────────────────────────────────────────
    @staticmethod
    def add_noise(
        engine:   GraphicEngine,
        strength: float = 0.08,
        monochrome: bool = True,
        seed:     int   = 0,
    ) -> GraphicEngine:
        W, H  = engine.width, engine.height
        rng   = np.random.default_rng(seed)
        noise = rng.integers(0, 255, (H, W), dtype=np.uint8)
        layer = np.zeros((H, W, 4), dtype=np.uint8)
        if monochrome:
            layer[:,:,0] = layer[:,:,1] = layer[:,:,2] = noise
        else:
            layer[:,:,0] = rng.integers(0, 255, (H, W), dtype=np.uint8)
            layer[:,:,1] = rng.integers(0, 255, (H, W), dtype=np.uint8)
            layer[:,:,2] = rng.integers(0, 255, (H, W), dtype=np.uint8)
        layer[:,:,3] = int(255 * clamp(strength, 0, 1))
        noise_img = Image.fromarray(layer, "RGBA")
        engine.canvas = Image.alpha_composite(engine.canvas, noise_img)
        engine._refresh_draw()
        return engine

    # ── Apply Effects Preset ──────────────────────────────────
    @staticmethod
    def apply_preset(
        engine: GraphicEngine,
        preset: str = "cinematic",
        **kwargs
    ) -> GraphicEngine:
        effects = EFFECT_PRESETS.get(preset, [])
        for fx in effects:
            try:
                if fx == "vignette":
                    VisualAestheticsEngine.add_vignette(
                        engine,
                        strength=kwargs.get("vignette_strength", 0.5)
                    )
                elif fx == "grain":
                    VisualAestheticsEngine.apply_texture(
                        engine, TextureType.GRAIN,
                        strength=kwargs.get("grain_strength", 0.1)
                    )
                elif fx == "bokeh":
                    VisualAestheticsEngine.add_bokeh_lights(
                        engine,
                        count=kwargs.get("bokeh_count", 12),
                        colors=kwargs.get("bokeh_colors", None)
                    )
                elif fx == "color_grade":
                    VisualAestheticsEngine.apply_color_grade(
                        engine,
                        preset=kwargs.get("grade_preset", "cinematic")
                    )
                elif fx == "texture":
                    VisualAestheticsEngine.apply_texture(
                        engine,
                        TextureType.PAPER,
                        strength=kwargs.get("texture_strength", 0.1)
                    )
            except Exception as e:
                print(f"  ⚠️ Effect '{fx}' skipped: {e}")
        return engine


# ── Quick Test ────────────────────────────────────────────────
print("╔══════════════════════════════════════════════════╗")
print("║    CELL 05 — Visual Aesthetics Engine ✅        ║")
print("╠══════════════════════════════════════════════════╣")
print("║  ✅ add_glassmorphism                           ║")
print("║  ✅ apply_texture (grain/paper/dots/lines/geo)  ║")
print("║  ✅ add_drop_shadow                             ║")
print("║  ✅ add_glow / add_text_glow                    ║")
print("║  ✅ add_vignette                                ║")
print("║  ✅ add_bokeh_lights (_refresh_draw fixed)      ║")
print("║  ✅ apply_color_grade                           ║")
print("║  ✅ add_noise                                   ║")
print("║  ✅ apply_preset                                ║")
print("╚══════════════════════════════════════════════════╝")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 06 — BRAND ENGINE + INTELLIGENT BRANDING
# Auto-color, brand apply, GRADIENT_PRESETS (Cell 07 fix)
# ══════════════════════════════════════════════════════════════

class BrandEngine:
    """
    Intelligent Brand ইঞ্জিন

    ✅ Auto-apply brand kit to GraphicEngine
    ✅ Gradient preset থেকে background তৈরি
    ✅ Color science (palette, harmony, contrast)
    ✅ Brand-consistent text styling
    """

    def __init__(self, kit_name: str = "ModernTech"):
        self.kit     = brand_manager.get_or_default(kit_name)
        self._engine: Optional[GraphicEngine] = None

    def set_kit(self, kit_name: str) -> 'BrandEngine':
        self.kit = brand_manager.get_or_default(kit_name)
        return self

    def apply_to(self, engine: GraphicEngine) -> GraphicEngine:
        """Brand kit background এবং settings apply করুন"""
        self._engine = engine
        engine.create_gradient_background(
            self.kit.background,
            blend_colors(self.kit.background, self.kit.primary, 0.5),
            GradientDirection.VERTICAL
        )
        return engine

    def heading(
        self, engine: GraphicEngine,
        text: str, y: int,
        font_size_key: str = "h1",
        color: Optional[str] = None,
        **kwargs
    ) -> Tuple:
        return engine.add_text(
            text, x='center', y=y,
            font_alias=self.kit.font_heading,
            font_size=FONT_SCALE.get(font_size_key, 80),
            color=color or self.kit.text_primary,
            **kwargs
        )

    def subheading(
        self, engine: GraphicEngine,
        text: str, y: int,
        font_size_key: str = "h3",
        color: Optional[str] = None,
        **kwargs
    ) -> Tuple:
        return engine.add_text(
            text, x='center', y=y,
            font_alias=self.kit.font_body,
            font_size=FONT_SCALE.get(font_size_key, 52),
            color=color or self.kit.text_secondary,
            **kwargs
        )

    def accent_bar(
        self, engine: GraphicEngine, y: int,
        width_pct: float = 0.15,
        thickness: int   = 5,
    ) -> GraphicEngine:
        return engine.add_accent_line(
            y, color=self.kit.accent,
            width_pct=width_pct, thickness=thickness
        )

    # ── Gradient Presets ──────────────────────────────────────
    @staticmethod
    def from_gradient_preset(
        engine:    GraphicEngine,
        preset:    str               = "midnight",
        direction: GradientDirection = GradientDirection.DIAGONAL,
    ) -> GraphicEngine:
        """GRADIENT_PRESETS থেকে background তৈরি"""
        if preset not in GRADIENT_PRESETS:
            available = list(GRADIENT_PRESETS.keys())
            print(f"  ⚠️ Preset '{preset}' নেই। Available: {available}")
            preset = "midnight"
        c1, c2 = GRADIENT_PRESETS[preset]
        engine.create_gradient_background(c1, c2, direction)
        return engine

    # ── Color Science ─────────────────────────────────────────
    @staticmethod
    def generate_palette(
        base_color: str,
        n: int = 5
    ) -> List[str]:
        """Base color থেকে harmonious palette তৈরি"""
        r, g, b = hex_to_rgb(base_color)
        palette = [base_color]
        for i in range(1, n):
            t = i / n
            # Lighten/darken alternately
            if i % 2 == 0:
                nr = int(r + (255-r)*t*0.6)
                ng = int(g + (255-g)*t*0.6)
                nb = int(b + (255-b)*t*0.6)
            else:
                nr = int(r * (1 - t*0.5))
                ng = int(g * (1 - t*0.5))
                nb = int(b * (1 - t*0.5))
            palette.append(rgb_to_hex(
                clamp(nr,0,255), clamp(ng,0,255), clamp(nb,0,255)
            ))
        return palette

    @staticmethod
    def complementary(color: str) -> str:
        r, g, b = hex_to_rgb(color)
        return rgb_to_hex(255-r, 255-g, 255-b)

    @staticmethod
    def analogous(color: str, n: int = 3) -> List[str]:
        """Analogous colors (hue shift)"""
        r, g, b = hex_to_rgb(color)
        results = []
        for i in range(n):
            shift = int(30 * (i - n//2))
            results.append(rgb_to_hex(
                clamp((r + shift) % 256, 0, 255),
                clamp(g, 0, 255),
                clamp((b - shift) % 256, 0, 255)
            ))
        return results

    @staticmethod
    def check_contrast(
        fg: str, bg: str,
        standard: str = "AA"
    ) -> Dict:
        ratio = contrast_ratio(fg, bg)
        thresholds = {
            "AAA": 7.0,
            "AA":  4.5,
            "AA_large": 3.0
        }
        return {
            "ratio":   round(ratio, 2),
            "AA":      ratio >= thresholds["AA"],
            "AAA":     ratio >= thresholds["AAA"],
            "AA_large": ratio >= thresholds["AA_large"],
            "grade":   "AAA" if ratio >= 7 else "AA" if ratio >= 4.5 else "AA_large" if ratio >= 3 else "FAIL",
        }

    @staticmethod
    def suggest_text_color(
        bg: str, options: List[str] = None
    ) -> str:
        """Background দেখে সবচেয়ে readable text color suggest করে"""
        opts = options or ["#FFFFFF", "#111111", "#F8FAFC", "#0F172A"]
        best, best_ratio = opts[0], 0
        for c in opts:
            r = contrast_ratio(c, bg)
            if r > best_ratio:
                best_ratio = r
                best = c
        return best


# ── Brand Engine Test ─────────────────────────────────────────
print("\n✅ BrandEngine ready")
print(f"   GRADIENT_PRESETS: {list(GRADIENT_PRESETS.keys())}")
print(f"   Presets available: {len(GRADIENT_PRESETS)}")

# Quick demo
with GraphicEngine(800, 300) as demo:
    BrandEngine.from_gradient_preset(demo, "neon", GradientDirection.DIAGONAL)
    VisualAestheticsEngine.add_bokeh_lights(
        demo, count=8, colors=["#6366F1","#EC4899"], seed=7
    )
    VisualAestheticsEngine.add_vignette(demo, strength=0.4)
    demo.add_text(
        "BrandEngine ✅  GRADIENT_PRESETS ✅",
        x='center', y=110,
        font_alias="bengali_bold", font_size=44,
        color="#FFFFFF", shadow=True
    )
    demo.show()

print("✅ Cell 06 ready — NameError: GRADIENT_PRESETS fixed ✅")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 07 — EXPORT ENGINE
# Platform sizes, smart resize, ZIP export, PDF export
# SyntaxError: '(' was never closed — সম্পূর্ণ fix করা হয়েছে
# ═══════════��══════════════════════════════════════════════════

PLATFORM_SIZES: Dict[str, Tuple[int, int]] = {
    # Instagram
    "instagram_square":    (1080, 1080),
    "instagram_portrait":  (1080, 1350),
    "instagram_landscape": (1080,  566),
    "instagram_story":     (1080, 1920),
    "instagram_reel":      (1080, 1920),
    # Facebook
    "facebook_post":       (1200,  630),
    "facebook_cover":      (1640,  624),
    "facebook_story":      (1080, 1920),
    "facebook_ad":         (1200,  628),
    # Twitter / X
    "twitter_post":        (1600,  900),
    "twitter_header":      (1500,  500),
    # YouTube
    "youtube_thumbnail":   (1280,  720),
    "youtube_banner":      (2560, 1440),
    # LinkedIn
    "linkedin_post":       (1200,  627),
    "linkedin_cover":      (1584,  396),
    # WhatsApp
    "whatsapp_status":     (1080, 1920),
    "whatsapp_dp":         ( 500,  500),
    # Print
    "a4_portrait":         (2480, 3508),
    "a4_landscape":        (3508, 2480),
    "business_card":       (1050,  600),
    "banner_horizontal":   (3000,  750),
    # Generic
    "square_sm":           ( 800,  800),
    "wide_sm":             (1200,  628),
    "story_portrait":      (1080, 1920),
}

PLATFORM_GROUPS: Dict[str, List[str]] = {
    "all_social": [
        "instagram_square", "instagram_story",
        "facebook_post", "twitter_post",
        "youtube_thumbnail", "linkedin_post",
    ],
    "instagram_all": [
        "instagram_square", "instagram_portrait",
        "instagram_landscape", "instagram_story", "instagram_reel",
    ],
    "facebook_all": [
        "facebook_post", "facebook_cover",
        "facebook_story", "facebook_ad",
    ],
    "print_all": [
        "a4_portrait", "a4_landscape",
        "business_card", "banner_horizontal",
    ],
    "quick": [
        "instagram_square", "instagram_story",
        "facebook_post", "youtube_thumbnail",
    ],
}


class ExportEngine:
    """Professional export — smart resize, multi-platform, ZIP, PDF"""

    # ── Smart Resize ──────────────────────────────────────────
    @staticmethod
    def smart_resize(
        engine: GraphicEngine,
        width:  int,
        height: int,
        mode:   str = "cover",
    ) -> GraphicEngine:
        """
        mode = 'cover'   → crop to fill
               'contain' → fit inside (letter-box)
               'stretch' → stretch exactly
        """
        src_w, src_h = engine.canvas.size
        src_r  = src_w / src_h
        tgt_r  = width / height

        if mode == "cover":
            if src_r > tgt_r:
                new_h = height
                new_w = int(src_w * height / src_h)
            else:
                new_w = width
                new_h = int(src_h * width / src_w)
            img  = engine.canvas.resize(
                (new_w, new_h), Image.Resampling.LANCZOS)
            left = (new_w - width)  // 2
            top  = (new_h - height) // 2
            img  = img.crop((left, top, left+width, top+height))

        elif mode == "contain":
            if src_r > tgt_r:
                new_w = width
                new_h = int(src_h * width / src_w)
            else:
                new_h = height
                new_w = int(src_w * height / src_h)
            base = Image.new("RGBA", (width, height), (0,0,0,255))
            img  = engine.canvas.resize(
                (new_w, new_h), Image.Resampling.LANCZOS)
            px = (width  - new_w) // 2
            py = (height - new_h) // 2
            base.paste(img, (px, py), img)
            img = base

        else:  # stretch
            img = engine.canvas.resize(
                (width, height), Image.Resampling.LANCZOS)

        out        = GraphicEngine(width, height)
        out.canvas = img
        out._refresh_draw()
        return out

    # ── Export single platform ────────────────────────────────
    @staticmethod
    def export_to_platform(
        engine:    GraphicEngine,
        platform:  str,
        name:      str = "export",
        fmt:       str = "PNG",
        directory: str = CONFIG.EXPORT_DIR,
    ) -> str:
        if platform not in PLATFORM_SIZES:
            raise ValueError(
                f"Unknown platform '{platform}'. "
                f"Available: {list(PLATFORM_SIZES.keys())}"
            )
        w, h    = PLATFORM_SIZES[platform]
        resized = ExportEngine.smart_resize(engine, w, h)
        os.makedirs(directory, exist_ok=True)
        fname = f"{name}_{platform}.{fmt.lower()}"
        path  = os.path.join(directory, fname)
        if fmt == "JPEG":
            resized.get_rgb().save(
                path, "JPEG",
                quality=CONFIG.JPEG_QUALITY, optimize=True
            )
        elif fmt == "WEBP":
            resized.get_rgb().save(
                path, "WEBP", quality=CONFIG.WEBP_QUALITY
            )
        else:
            resized.canvas.save(path, "PNG")
        kb = os.path.getsize(path) // 1024
        print(f"  ✅ {platform:<28} {w}×{h}  {kb}KB")
        return path

    # ── Export platform group ─────────────────────────────────
    @staticmethod
    def export_platform_group(
        engine:     GraphicEngine,
        group:      str  = "all_social",
        output_dir: Optional[str] = None,
        prefix:     str  = "",
        fmt:        str  = "PNG",
        create_zip: bool = True,
    ) -> Dict[str, str]:
        platforms  = PLATFORM_GROUPS.get(group, list(PLATFORM_SIZES.keys()))
        directory  = output_dir or os.path.join(
            CONFIG.EXPORT_DIR, f"{prefix}platforms"
        )
        os.makedirs(directory, exist_ok=True)
        results: Dict[str, str] = {}

        print(f"\n  🚀 Platform Group Export")
        print(f"  {'─'*48}")
        print(f"  📐 Group : {group} ({len(platforms)} platforms)")
        print(f"  💾 Output: {directory}")
        print(f"  {'─'*48}")

        bar = tqdm(platforms, desc="  ⚡ Resizing")
        for platform in bar:
            try:
                w, h    = PLATFORM_SIZES[platform]
                resized = ExportEngine.smart_resize(engine, w, h)
                fname   = f"{platform}.{fmt.lower()}"
                path    = os.path.join(directory, fname)
                if fmt == "JPEG":
                    resized.get_rgb().save(
                        path, "JPEG", quality=CONFIG.JPEG_QUALITY)
                elif fmt == "WEBP":
                    resized.get_rgb().save(
                        path, "WEBP", quality=CONFIG.WEBP_QUALITY)
                else:
                    resized.canvas.save(path, "PNG")
                kb = os.path.getsize(path) // 1024
                bar.set_postfix({"platform": platform, "kb": kb})
                results[platform] = path
            except Exception as e:
                print(f"  ❌ {platform}: {e}")

        print(f"  ✅ {len(results)} platforms exported")

        # ✅ SyntaxError fix — parenthesis সঠিকভাবে বন্ধ
        if create_zip and results:
            zip_path = os.path.join(
                CONFIG.EXPORT_DIR,
                f"{prefix}{group}.zip"
            )
            with zipfile.ZipFile(
                zip_path,
                mode="w",
                compression=zipfile.ZIP_DEFLATED,
                compresslevel=CONFIG.ZIP_COMPRESSION,
            ) as zf:
                for plat, fpath in results.items():
                    zf.write(fpath, os.path.basename(fpath))
            kb = os.path.getsize(zip_path) // 1024
            print(f"  📦 ZIP: {zip_path} ({kb}KB)")
            results["__zip__"] = zip_path

        return results

    # ── Export multi-format ───────────────────────────────────
    @staticmethod
    def export_multi_format(
        engine:    GraphicEngine,
        name:      str,
        formats:   Optional[List[str]] = None,
        directory: str = CONFIG.OUTPUT_DIR,
    ) -> Dict[str, str]:
        formats = formats or ["PNG", "JPEG", "WEBP"]
        results: Dict[str, str] = {}
        os.makedirs(directory, exist_ok=True)
        for fmt in formats:
            try:
                ext  = fmt.lower()
                path = os.path.join(directory, f"{name}.{ext}")
                if fmt == "JPEG":
                    engine.get_rgb().save(
                        path, "JPEG",
                        quality=CONFIG.JPEG_QUALITY, optimize=True
                    )
                elif fmt == "WEBP":
                    engine.get_rgb().save(
                        path, "WEBP", quality=CONFIG.WEBP_QUALITY
                    )
                else:
                    engine.canvas.save(path, "PNG")
                results[fmt] = path
            except Exception as e:
                print(f"  ❌ {fmt}: {e}")
        return results

    # ── Export PDF ────────────────────────────────────────────
    @staticmethod
    def export_pdf(
        images:    List[Image.Image],
        filename:  str,
        directory: str = CONFIG.EXPORT_DIR,
    ) -> str:
        if not FPDF_AVAILABLE:
            print("⚠️  fpdf2 not installed — PDF export skipped")
            return ""
        os.makedirs(directory, exist_ok=True)
        os.makedirs(CONFIG.TEMP_DIR, exist_ok=True)
        path = os.path.join(directory, filename)
        pdf  = FPDF()
        pdf.set_auto_page_break(False)
        for i, img in enumerate(images):
            w_mm = img.width  * 25.4 / 96
            h_mm = img.height * 25.4 / 96
            pdf.add_page(
                format=(w_mm, h_mm),
                orientation='P' if h_mm >= w_mm else 'L'
            )
            tmp = os.path.join(CONFIG.TEMP_DIR, f"_tmp_page_{i}.jpg")
            img.convert("RGB").save(
                tmp, "JPEG", quality=CONFIG.JPEG_QUALITY)
            pdf.image(tmp, 0, 0, w_mm, h_mm)
        pdf.output(path)
        kb = os.path.getsize(path) // 1024
        print(f"  📄 PDF: {path} ({kb}KB)")
        return path


# ── Status ────────────────────────────────────────────────────
print("╔══════════════════════════════════════════════════╗")
print("║      CELL 07 — Export Engine ✅                 ║")
print("╠══════════════════════════════════════════════════╣")
print(f"║  📐 Platform Sizes  : {len(PLATFORM_SIZES):<3}                    ║")
print(f"║  📦 Platform Groups : {len(PLATFORM_GROUPS):<3}                    ║")
print("║  ✅ SyntaxError fix : ZipFile() closed         ║")
print("║  ✅ smart_resize    : cover/contain/stretch    ║")
print("║  ✅ export_to_platform                        ║")
print("║  ✅ export_platform_group + ZIP               ║")
print("║  ✅ export_multi_format                       ║")
print("║  ✅ export_pdf                                ║")
print("╚══════════════════════════════════════════════════╝")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 08 — AI PROCESSING ENGINE
# CV2_AVAILABLE NameError fix + rembg + image enhancement
# ══════════════════════════════════════════════════════════════

class AIProcessingEngine:
    """
    AI Image Processing ইঞ্জিন

    ✅ Background removal (rembg / OpenCV GrabCut)
    ✅ Smart crop (face-aware)
    ✅ Image enhancement
    ✅ Object detection (basic)
    ✅ Color extraction
    ✅ Auto resize with content-awareness
    """

    # ── Background Removal ────────────────────────────────────
    @staticmethod
    def remove_bg(
        img_input: Union[bytes, Image.Image],
        method:    str = "auto",
    ) -> Image.Image:
        """
        method = 'auto'   → rembg if available, else grabcut
                 'rembg'  → rembg (দরকার: pip install rembg)
                 'grabcut' → OpenCV GrabCut
        """
        # auto-select
        if method == "auto":
            method = "rembg" if REMBG_AVAILABLE else "grabcut"

        if method == "rembg" and REMBG_AVAILABLE:
            try:
                import rembg as _rb
                if isinstance(img_input, Image.Image):
                    buf = io.BytesIO()
                    img_input.save(buf, "PNG")
                    img_input = buf.getvalue()
                result = _rb.remove(img_input)
                return Image.open(io.BytesIO(result)).convert("RGBA")
            except Exception as e:
                print(f"  ⚠️ rembg failed: {e} — falling back to grabcut")

        # GrabCut fallback — ✅ CV2_AVAILABLE check সঠিকভাবে
        if CV2_AVAILABLE:
            return remove_background(img_input)

        # Last resort
        if isinstance(img_input, bytes):
            return Image.open(io.BytesIO(img_input)).convert("RGBA")
        return img_input.convert("RGBA") if hasattr(img_input, 'convert') \
               else Image.new("RGBA", (200,200), (0,0,0,0))

    # ── Smart Crop ────────────────────────────────────────────
    @staticmethod
    def smart_crop(
        img:         Image.Image,
        target_w:    int,
        target_h:    int,
        focus:       str = "center",
    ) -> Image.Image:
        """
        focus = 'center' | 'top' | 'bottom' | 'left' | 'right'
        """
        src_w, src_h = img.size
        src_r = src_w / src_h
        tgt_r = target_w / target_h

        if src_r > tgt_r:
            new_h = target_h
            new_w = int(src_w * target_h / src_h)
        else:
            new_w = target_w
            new_h = int(src_h * target_w / src_w)

        img = img.resize((new_w, new_h), Image.Resampling.LANCZOS)

        excess_w = new_w - target_w
        excess_h = new_h - target_h

        focus_map = {
            "center": (0.5, 0.5),
            "top":    (0.5, 0.0),
            "bottom": (0.5, 1.0),
            "left":   (0.0, 0.5),
            "right":  (1.0, 0.5),
        }
        fx, fy = focus_map.get(focus, (0.5, 0.5))
        left   = int(excess_w * fx)
        top    = int(excess_h * fy)
        return img.crop((left, top, left+target_w, top+target_h))

    # ── Image Enhancement ─────────────────────────────────────
    @staticmethod
    def enhance(
        img:        Image.Image,
        sharpness:  float = 1.0,
        saturation: float = 1.0,
        contrast:   float = 1.0,
        brightness: float = 1.0,
    ) -> Image.Image:
        if sharpness != 1.0:
            img = ImageEnhance.Sharpness(img).enhance(sharpness)
        if saturation != 1.0:
            img = ImageEnhance.Color(img).enhance(saturation)
        if contrast != 1.0:
            img = ImageEnhance.Contrast(img).enhance(contrast)
        if brightness != 1.0:
            img = ImageEnhance.Brightness(img).enhance(brightness)
        return img

    # ── Auto Enhance ──────────────────────────────────────────
    @staticmethod
    def auto_enhance(
        img: Image.Image,
        preset: str = "natural",
    ) -> Image.Image:
        presets = {
            "natural":   {"sharpness":1.1, "saturation":1.05, "contrast":1.02},
            "vivid":     {"sharpness":1.2, "saturation":1.3,  "contrast":1.1},
            "portrait":  {"sharpness":0.9, "saturation":0.95, "contrast":0.98},
            "landscape": {"sharpness":1.3, "saturation":1.2,  "contrast":1.1},
            "product":   {"sharpness":1.4, "saturation":1.0,  "contrast":1.15},
        }
        p = presets.get(preset, presets["natural"])
        return AIProcessingEngine.enhance(img, **p)

    # ── Dominant Colors ───────────────────────────────────────
    @staticmethod
    def get_colors(
        img_bytes: bytes,
        n: int = 5
    ) -> List[str]:
        return extract_dominant_colors(img_bytes, n_colors=n)

    # ── Image to bytes ────────────────────────────────────────
    @staticmethod
    def to_bytes(
        img: Image.Image,
        fmt: str = "PNG",
        quality: int = CONFIG.JPEG_QUALITY,
    ) -> bytes:
        buf = io.BytesIO()
        if fmt == "JPEG":
            img.convert("RGB").save(buf, "JPEG", quality=quality)
        else:
            img.save(buf, fmt)
        return buf.getvalue()

    # ── Watermark ─────────────────────────────────────────────
    @staticmethod
    def add_watermark(
        engine:      GraphicEngine,
        text:        str   = "© ColabCanvas",
        position:    str   = "bottom-right",
        font_alias:  str   = "english_regular",
        font_size:   int   = 18,
        color:       str   = "#FFFFFF",
        opacity:     float = 0.4,
        margin:      int   = 30,
    ) -> GraphicEngine:
        font = font_manager.get(font_alias, font_size)
        tmp  = ImageDraw.Draw(Image.new("RGBA", (1,1)))
        bb   = tmp.textbbox((0,0), text, font=font)
        tw   = bb[2] - bb[0]
        th   = bb[3] - bb[1]
        W, H = engine.width, engine.height

        if "right" in position:
            x = W - tw - margin
        elif "left" in position:
            x = margin
        else:
            x = (W - tw) // 2

        if "bottom" in position:
            y = H - th - margin
        elif "top" in position:
            y = margin
        else:
            y = (H - th) // 2

        engine.add_text(
            text, x=x, y=y,
            font_alias=font_alias, font_size=font_size,
            color=color, alpha=clamp(opacity, 0, 1)
        )
        return engine


# ── Status ────────────────────────────────────────────────────
print("╔══════════════════════════════════════════════════╗")
print("║     CELL 08 — AI Processing Engine ✅           ║")
print("╠══════════════════════════════════════════════════╣")
print(f"║  🤖 rembg Available    : {'✅' if REMBG_AVAILABLE else '❌ (install rembg)'}           ║")
print(f"║  👁️  OpenCV Available   : {'✅' if CV2_AVAILABLE else '❌'}                  ║")
print("║  ✅ CV2_AVAILABLE NameError — fixed!            ║")
print("║  ✅ remove_bg (auto/rembg/grabcut)              ║")
print("║  ✅ smart_crop                                  ║")
print("║  ✅ enhance / auto_enhance                      ║")
print("║  ✅ get_colors                                  ║")
print("║  ✅ add_watermark                               ║")
print("╚══════════════════════════════════════════════════╝")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 09 — TEMPLATE ENGINE
# 8+ templates — add_divider_line AttributeError fix
# ══════════════════════════════════════════════════════════════

class TemplateEngine:
    """
    Professional Template Library

    Templates:
    ──────────
    01. product_promo
    02. webinar_flyer
    03. instagram_story
    04. sale_announcement
    05. quote_card
    06. youtube_thumbnail
    07. facebook_cover
    08. bengali_special
    09. event_poster
    10. minimal_card
    """

    _registry: Dict[str, Any] = {}

    @classmethod
    def register(cls, name: str):
        def decorator(fn):
            cls._registry[name] = fn
            return fn
        return decorator

    @classmethod
    def list_templates(cls) -> List[str]:
        return sorted(cls._registry.keys())

    @classmethod
    def build(
        cls,
        template: str,
        params:   Dict[str, Any],
        show:     bool = True,
        save:     bool = True,
    ) -> GraphicEngine:
        if template not in cls._registry:
            avail = cls.list_templates()
            raise KeyError(
                f"Template '{template}' নেই.\n"
                f"Available: {avail}"
            )
        try:
            eng = cls._registry[template](**params)
            if show:
                eng.show()
            if save:
                fname = f"{template}_{datetime.now().strftime('%H%M%S')}.png"
                path  = eng.save(fname)
                print(f"  💾 Saved: {path}")
            return eng
        except Exception as e:
            traceback.print_exc()
            raise RuntimeError(f"Template '{template}' build failed: {e}") from e


# ══════════════════════════════════════════════════════════════
# TEMPLATE DEFINITIONS
# ══════════════════════════════════════════════════════════════

@TemplateEngine.register("product_promo")
def product_promo(
    headline:      str  = "নতুন পণ্য এসেছে!",
    subheadline:   str  = "অবিশ্বাস্য মূল্যে পাওয়া যাচ্ছে",
    price:         str  = "৳ ৯৯৯",
    cta:           str  = "এখনই অর্ডার করুন",
    kit_name:      str  = "ModernTech",
    bg_image_bytes: Optional[bytes] = None,
    product_bytes: Optional[bytes]  = None,
    qr_link:       Optional[str]    = None,
    effects:       Optional[Dict]   = None,
) -> GraphicEngine:
    kit = brand_manager.get_or_default(kit_name)
    eng = GraphicEngine(1080, 1080)
    effects = effects or {}

    if bg_image_bytes:
        eng.set_background_image(
            bg_image_bytes,
            blur=12, brightness=0.6,
            overlay_color=kit.background, overlay_alpha=0.55
        )
    else:
        eng.create_gradient_background(
            kit.background,
            blend_colors(kit.background, kit.primary, 0.6),
            GradientDirection.DIAGONAL
        )
        VisualAestheticsEngine.add_bokeh_lights(
            eng, count=10, colors=[kit.accent, kit.primary]
        )

    # ── Product image ─────────────────────────────────────────
    if product_bytes:
        eng.paste_image(
            product_bytes, x='center', y=180,
            width=500, height=420,
            shape=ImageShape.ROUNDED_SQUARE
        )

    # ── Price panel ───────────────────────────────────────────
    panel_y = 620
    eng.add_rectangle(
        60, panel_y, 1020, panel_y+340,
        fill=kit.primary, radius=24, alpha=0.88
    )

    # ── Divider ───────────────────────────────────────────────
    # ✅ add_divider_line alias — AttributeError fix
    eng.add_divider_line(
        y=panel_y + 10,
        color=kit.accent, thickness=3, alpha=0.8,
        margin=80
    )

    # ── Texts ─────────────────────────────────────────────────
    eng.add_text(
        headline, x='center', y=panel_y + 30,
        font_alias=kit.font_heading,
        font_size=FONT_SCALE["h2"], color=kit.text_primary,
        shadow=True
    )
    eng.add_text(
        subheadline, x='center', y=panel_y + 115,
        font_alias=kit.font_body,
        font_size=FONT_SCALE["body_lg"], color=kit.text_secondary
    )
    eng.add_accent_line(
        panel_y + 175, color=kit.accent,
        width_pct=0.2, thickness=4
    )
    eng.add_text(
        price, x='center', y=panel_y + 195,
        font_alias=kit.font_heading,
        font_size=FONT_SCALE["price"], color=kit.accent,
        shadow=True
    )

    # ── CTA button ────────────────────────────────────────────
    btn_w = 480
    btn_x = (1080 - btn_w) // 2
    eng.add_rectangle(
        btn_x, panel_y + 290, btn_x + btn_w, panel_y + 320,
        fill=kit.accent, radius=30, alpha=1.0
    )
    eng.add_text(
        cta, x='center', y=panel_y + 298,
        font_alias=kit.font_heading,
        font_size=FONT_SCALE["label_lg"],
        color=auto_text_color(kit.accent)
    )

    # ── QR ────────────────────────────────────────────────────
    if qr_link:
        eng.add_qr_code(
            qr_link, x=900, y=900, size=140,
            fill_color=kit.accent, back_color="#FFFFFF"
        )

    VisualAestheticsEngine.add_vignette(eng, strength=0.45)
    return eng


@TemplateEngine.register("webinar_flyer")
def webinar_flyer(
    title:        str  = "ডিজিটাল মার্কেটিং ওয়েবিনার",
    subtitle:     str  = "আপনার ব্যবসাকে অনলাইনে নিয়ে যান",
    date:         str  = "২০ ফেব্রুয়ারি ২০২৬",
    time:         str  = "সন্ধ্যা ৮:০০ টা",
    speaker:      str  = "মোহাম্মদ রাহেল",
    platform:     str  = "Zoom • Facebook Live",
    kit_name:     str  = "VibrantCreative",
    host_bytes:   Optional[bytes] = None,
    qr_link:      Optional[str]   = None,
) -> GraphicEngine:
    kit = brand_manager.get_or_default(kit_name)
    eng = GraphicEngine(1080, 1080)

    # Background
    eng.create_gradient_background(
        kit.background, kit.primary, GradientDirection.DIAGONAL
    )
    VisualAestheticsEngine.add_bokeh_lights(
        eng, count=12, colors=[kit.accent, kit.secondary], seed=5
    )
    VisualAestheticsEngine.add_vignette(eng, strength=0.5)

    # Header strip
    eng.add_rectangle(0, 0, 1080, 120,
                      fill=kit.primary, alpha=0.85)
    eng.add_text(
        "WEBINAR", x='center', y=30,
        font_alias="english_bold",
        font_size=FONT_SCALE["overline"],
        color=kit.accent
    )
    eng.add_text(
        platform, x='center', y=72,
        font_alias="english_regular",
        font_size=FONT_SCALE["caption"],
        color=kit.text_secondary
    )

    # Host image
    if host_bytes:
        eng.paste_image(
            host_bytes, x='center', y=145,
            width=280, height=280,
            shape=ImageShape.CIRCLE,
            outline_color=kit.accent, outline_width=6
        )
        speaker_y = 455
    else:
        speaker_y = 170

    # Title
    eng.add_text(
        title, x='center', y=speaker_y,
        font_alias=kit.font_heading,
        font_size=FONT_SCALE["h2"], color=kit.text_primary,
        shadow=True, max_width=900
    )
    eng.add_accent_line(
        speaker_y + 120, color=kit.accent,
        width_pct=0.3, thickness=5
    )
    eng.add_text(
        subtitle, x='center', y=speaker_y + 140,
        font_alias=kit.font_body,
        font_size=FONT_SCALE["body_lg"], color=kit.text_secondary,
        max_width=860
    )

    # Speaker
    eng.add_rectangle(
        240, speaker_y + 230, 840, speaker_y + 285,
        fill=kit.accent, radius=8, alpha=0.2
    )
    eng.add_text(
        f"🎤 {speaker}", x='center', y=speaker_y + 238,
        font_alias=kit.font_body,
        font_size=FONT_SCALE["body_lg"], color=kit.accent
    )

    # Date/Time info
    eng.add_divider_line(
        speaker_y + 310,
        color=kit.text_secondary, alpha=0.3, thickness=1
    )
    eng.add_text(
        f"📅  {date}", x='center', y=speaker_y + 330,
        font_alias=kit.font_body,
        font_size=FONT_SCALE["body"], color=kit.text_primary
    )
    eng.add_text(
        f"⏰  {time}", x='center', y=speaker_y + 385,
        font_alias=kit.font_body,
        font_size=FONT_SCALE["body"], color=kit.text_primary
    )

    # QR
    if qr_link:
        eng.add_qr_code(
            qr_link, x=910, y=910, size=130,
            fill_color="#FFFFFF", back_color="#000000"
        )

    return eng


@TemplateEngine.register("instagram_story")
def instagram_story(
    headline:    str  = "আজকের বিশেষ অফার",
    body_text:   str  = "সীমিত সময়ের জন্য বিশেষ ছাড়",
    cta:         str  = "স্ক্রোল আপ করুন ↑",
    kit_name:    str  = "DarkNeon",
    image_bytes: Optional[bytes] = None,
    effects:     Optional[List[str]] = None,
) -> GraphicEngine:
    kit     = brand_manager.get_or_default(kit_name)
    eng     = GraphicEngine(1080, 1920)
    effects = effects or ["vignette", "bokeh"]

    if image_bytes:
        eng.set_background_image(
            image_bytes, blur=8, brightness=0.5,
            overlay_color=kit.background, overlay_alpha=0.6
        )
    else:
        eng.create_gradient_background(
            kit.background, kit.primary, GradientDirection.VERTICAL
        )

    if "bokeh" in effects:
        VisualAestheticsEngine.add_bokeh_lights(
            eng, count=18,
            colors=[kit.accent, kit.primary, kit.secondary]
        )

    # Top decoration
    eng.add_rectangle(0, 0, 1080, 80,
                      fill=kit.accent, alpha=0.15)
    eng.add_text(
        "SWIPE UP", x='center', y=22,
        font_alias="english_bold",
        font_size=FONT_SCALE["overline"],
        color=kit.accent
    )

    # Main content panel
    eng.add_rectangle(
        60, 700, 1020, 1580,
        fill=kit.primary, radius=30, alpha=0.88
    )

    # Glassmorphism panel at bottom
    VisualAestheticsEngine.add_glassmorphism(
        eng, 60, 700, 1020, 900,
        blur_radius=15, fill_opacity=0.12,
        border_color=kit.accent, border_opacity=0.3,
        radius=30
    )

    eng.add_text(
        headline, x='center', y=740,
        font_alias=kit.font_heading,
        font_size=FONT_SCALE["h1"], color=kit.text_primary,
        shadow=True, max_width=900
    )
    eng.add_accent_line(
        860, color=kit.accent,
        width_pct=0.25, thickness=5
    )
    eng.add_text(
        body_text, x='center', y=900,
        font_alias=kit.font_body,
        font_size=FONT_SCALE["h3"], color=kit.text_secondary,
        max_width=860
    )

    # CTA strip
    eng.add_rectangle(
        0, 1820, 1080, 1920,
        fill=kit.accent, alpha=1.0
    )
    eng.add_text(
        cta, x='center', y=1845,
        font_alias=kit.font_heading,
        font_size=FONT_SCALE["body_lg"],
        color=auto_text_color(kit.accent)
    )

    if "vignette" in effects:
        VisualAestheticsEngine.add_vignette(eng, strength=0.5)

    return eng


@TemplateEngine.register("sale_announcement")
def sale_announcement(
    percent:     str  = "৫০%",
    label:       str  = "ছাড়!",
    subtitle:    str  = "সীমিত সময়ের অফার",
    code:        str  = "SALE50",
    validity:    str  = "৩১ মার্চ পর্যন্ত",
    kit_name:    str  = "BoldCorporate",
    image_bytes: Optional[bytes] = None,
) -> GraphicEngine:
    kit = brand_manager.get_or_default(kit_name)
    eng = GraphicEngine(1080, 1080)

    if image_bytes:
        eng.set_background_image(
            image_bytes, blur=20, brightness=0.4,
            overlay_color=kit.background, overlay_alpha=0.7
        )
    else:
        eng.create_gradient_background(
            kit.background, kit.secondary, GradientDirection.RADIAL
        )

    VisualAestheticsEngine.add_bokeh_lights(
        eng, count=8, colors=[kit.accent], seed=99
    )

    # Burst decoration
    for i, (size, alpha) in enumerate([(600,0.05),(450,0.07),(300,0.1)]):
        eng.add_circle(540, 540, size//2,
                       fill=kit.accent, alpha=alpha)

    eng.add_rectangle(
        80, 200, 1000, 880,
        fill="#000000", radius=28, alpha=0.6
    )

    eng.add_text(
        percent, x='center', y=230,
        font_alias=kit.font_heading,
        font_size=FONT_SCALE["numeral_lg"], color=kit.accent,
        shadow=True
    )
    eng.add_text(
        label, x='center', y=420,
        font_alias=kit.font_heading,
        font_size=FONT_SCALE["display"], color=kit.text_primary,
        shadow=True
    )
    eng.add_divider_line(
        540, color=kit.accent, thickness=3, alpha=0.7
    )
    eng.add_text(
        subtitle, x='center', y=565,
        font_alias=kit.font_body,
        font_size=FONT_SCALE["h3"], color=kit.text_secondary
    )

    # Coupon code box
    eng.add_rectangle(
        240, 660, 840, 740,
        fill=kit.accent, radius=12, alpha=1.0
    )
    eng.add_text(
        f"কোড:  {code}", x='center', y=672,
        font_alias="english_bold",
        font_size=FONT_SCALE["h4"],
        color=auto_text_color(kit.accent)
    )
    eng.add_text(
        validity, x='center', y=765,
        font_alias=kit.font_body,
        font_size=FONT_SCALE["caption"],
        color=kit.text_secondary
    )

    VisualAestheticsEngine.add_vignette(eng, strength=0.6)
    return eng


@TemplateEngine.register("quote_card")
def quote_card(
    quote:      str  = "স্বপ্ন দেখুন, পরিশ্রম করুন, সাফল্য পান।",
    author:     str  = "— অজানা",
    category:   str  = "অনুপ্রেরণা",
    kit_name:   str  = "ElegantMinimal",
    image_bytes: Optional[bytes] = None,
) -> GraphicEngine:
    kit = brand_manager.get_or_default(kit_name)
    eng = GraphicEngine(1080, 1080)

    if image_bytes:
        eng.set_background_image(
            image_bytes, blur=25, brightness=0.5,
            overlay_color=kit.background, overlay_alpha=0.75
        )
    else:
        eng.create_gradient_background(
            kit.background, kit.primary, GradientDirection.DIAGONAL
        )

    # Category label
    eng.add_text(
        category.upper(), x='center', y=80,
        font_alias="english_italic",
        font_size=FONT_SCALE["overline"], color=kit.accent
    )

    # Large quotation mark
    eng.add_text(
        "\u201c", x=100, y=160,
        font_alias=kit.font_heading,
        font_size=200, color=kit.accent, alpha=0.25
    )

    # Glass card
    VisualAestheticsEngine.add_glassmorphism(
        eng, 100, 280, 980, 780,
        blur_radius=18, fill_opacity=0.12,
        border_color=kit.accent, border_opacity=0.2,
        radius=24
    )

    # Quote text
    eng.add_text(
        quote, x='center', y=340,
        font_alias=kit.font_heading,
        font_size=FONT_SCALE["h3"], color=kit.text_primary,
        max_width=820, line_spacing=20, shadow=True
    )
    eng.add_accent_line(
        700, color=kit.accent,
        width_pct=0.15, align="center", thickness=3
    )
    eng.add_text(
        author, x='center', y=730,
        font_alias="english_italic",
        font_size=FONT_SCALE["body_lg"], color=kit.text_secondary
    )

    VisualAestheticsEngine.add_vignette(eng, strength=0.4)
    return eng


@TemplateEngine.register("youtube_thumbnail")
def youtube_thumbnail(
    title:       str  = "এই ভিডিওতে সব কিছু শিখুন!",
    label:       str  = "TUTORIAL",
    presenter:   str  = "",
    kit_name:    str  = "ModernTech",
    image_bytes: Optional[bytes] = None,
    face_bytes:  Optional[bytes] = None,
) -> GraphicEngine:
    kit = brand_manager.get_or_default(kit_name)
    eng = GraphicEngine(1280, 720)

    if image_bytes:
        eng.set_background_image(
            image_bytes, blur=4, brightness=0.55,
            overlay_color=kit.background, overlay_alpha=0.55
        )
    else:
        eng.create_gradient_background(
            kit.background, kit.secondary, GradientDirection.HORIZONTAL
        )

    VisualAestheticsEngine.add_bokeh_lights(
        eng, count=8, colors=[kit.accent, kit.primary], seed=11
    )

    # Face / presenter
    if face_bytes:
        eng.paste_image(
            face_bytes, x=780, y='center',
            width=400, height=480,
            shape=ImageShape.ROUNDED_SQUARE
        )

    # Label badge
    eng.add_rectangle(
        60, 60, 60 + len(label)*22 + 40, 115,
        fill=kit.accent, radius=8, alpha=1.0
    )
    eng.add_text(
        label, x=80, y=68,
        font_alias="english_bold",
        font_size=FONT_SCALE["overline"],
        color=auto_text_color(kit.accent)
    )

    # Title
    eng.add_text(
        title, x=60, y=180,
        font_alias=kit.font_heading,
        font_size=FONT_SCALE["h2"], color=kit.text_primary,
        max_width=680, shadow=True
    )

    if presenter:
        eng.add_text(
            presenter, x=60, y=560,
            font_alias="english_regular",
            font_size=FONT_SCALE["body"], color=kit.text_secondary
        )

    # Accent left bar
    eng.add_rectangle(
        0, 0, 8, 720,
        fill=kit.accent, alpha=1.0
    )

    VisualAestheticsEngine.add_vignette(eng, strength=0.35)
    return eng


@TemplateEngine.register("facebook_cover")
def facebook_cover(
    brand_name:  str  = "আমার ব্র্যান্ড",
    tagline:     str  = "সেরা মানের সেরা পণ্য",
    website:     str  = "www.example.com",
    kit_name:    str  = "BengaliVibrant",
    image_bytes: Optional[bytes] = None,
    logo_bytes:  Optional[bytes] = None,
) -> GraphicEngine:
    kit = brand_manager.get_or_default(kit_name)
    eng = GraphicEngine(1640, 624)

    if image_bytes:
        eng.set_background_image(
            image_bytes, blur=6, brightness=0.55,
            overlay_color=kit.background, overlay_alpha=0.6
        )
    else:
        eng.create_gradient_background(
            kit.background, kit.primary, GradientDirection.HORIZONTAL
        )

    # Logo
    if logo_bytes:
        eng.paste_image(
            logo_bytes, x=120, y='center',
            width=220, height=220,
            shape=ImageShape.CIRCLE,
            outline_color=kit.accent, outline_width=5
        )
        text_x = 380
    else:
        text_x = 120

    eng.add_text(
        brand_name, x=text_x, y=200,
        font_alias=kit.font_heading,
        font_size=FONT_SCALE["h1"], color=kit.text_primary,
        shadow=True
    )
    eng.add_accent_line(
        310, color=kit.accent,
        width_pct=0.35, align="left", thickness=4
    )
    eng.add_text(
        tagline, x=text_x, y=335,
        font_alias=kit.font_body,
        font_size=FONT_SCALE["h4"], color=kit.text_secondary
    )
    eng.add_text(
        website, x=text_x, y=440,
        font_alias="english_regular",
        font_size=FONT_SCALE["body"], color=kit.accent
    )

    VisualAestheticsEngine.add_vignette(eng, strength=0.35)
    return eng


@TemplateEngine.register("bengali_special")
def bengali_special(
    headline:   str  = "বিশেষ ঘোষণা",
    subheadline: str = "আমাদের নতুন সেবা চালু হলো",
    body:       str  = "আজ থেকেই উপভোগ করুন আমাদের অসাধারণ সেবা।",
    footer:     str  = "যোগাযোগ: ০১৭০০-০০০০০০",
    kit_name:   str  = "BengaliVibrant",
    image_bytes: Optional[bytes] = None,
) -> GraphicEngine:
    kit = brand_manager.get_or_default(kit_name)
    eng = GraphicEngine(1080, 1080)

    if image_bytes:
        eng.set_background_image(
            image_bytes, blur=14, brightness=0.45,
            overlay_color=kit.background, overlay_alpha=0.7
        )
    else:
        eng.create_gradient_background(
            kit.background, kit.secondary, GradientDirection.VERTICAL
        )
    VisualAestheticsEngine.add_bokeh_lights(
        eng, count=10, colors=[kit.accent, kit.primary], seed=3
    )

    # Top accent band
    eng.add_rectangle(0, 0, 1080, 14, fill=kit.accent, alpha=1.0)

    # Decorative corner
    eng.add_corner_decoration(kit.accent, 180, "top-right", "triangle", 60)
    eng.add_corner_decoration(kit.secondary, 120, "bottom-left", "triangle", 40)

    # Main content
    eng.add_text(
        "✦", x='center', y=80,
        font_alias="english_regular",
        font_size=50, color=kit.accent, alpha=0.7
    )
    eng.add_text(
        headline, x='center', y=160,
        font_alias=kit.font_heading,
        font_size=FONT_SCALE["bengali_h1"], color=kit.text_primary,
        shadow=True
    )
    eng.add_accent_line(
        290, color=kit.accent,
        width_pct=0.3, thickness=5
    )
    eng.add_text(
        subheadline, x='center', y=320,
        font_alias=kit.font_body,
        font_size=FONT_SCALE["h3"], color=kit.text_secondary,
        max_width=880
    )

    # Body text card
    eng.add_rectangle(
        100, 430, 980, 660,
        fill=kit.primary, radius=20, alpha=0.82
    )
    eng.add_text(
        body, x='center', y=470,
        font_alias=kit.font_body,
        font_size=FONT_SCALE["bengali_body"], color=kit.text_primary,
        max_width=800, line_spacing=18
    )

    # Footer
    eng.add_rectangle(0, 980, 1080, 1080, fill=kit.primary, alpha=0.9)
    eng.add_text(
        footer, x='center', y=1018,
        font_alias=kit.font_body,
        font_size=FONT_SCALE["body"], color=kit.accent
    )

    # Bottom accent band
    eng.add_rectangle(0, 1066, 1080, 1080, fill=kit.accent, alpha=1.0)
    VisualAestheticsEngine.add_vignette(eng, strength=0.45)
    return eng


@TemplateEngine.register("event_poster")
def event_poster(
    event_name:  str = "বার্ষিক সাংস্কৃতিক উৎসব ২০২৬",
    date:        str = "১৫ মার্চ ২০২৬",
    time:        str = "বিকেল ৪:০০ টা",
    venue:       str = "জাতীয় শিল্পকলা একাডেমি, ঢাকা",
    ticket:      str = "প্রবেশ মূল্য: ৳ ২০০",
    kit_name:    str = "VibrantCreative",
    image_bytes: Optional[bytes] = None,
) -> GraphicEngine:
    kit = brand_manager.get_or_default(kit_name)
    eng = GraphicEngine(1080, 1350)

    if image_bytes:
        eng.set_background_image(
            image_bytes, blur=10, brightness=0.45,
            overlay_color=kit.background, overlay_alpha=0.65
        )
    else:
        eng.create_gradient_background(
            kit.background, kit.primary, GradientDirection.DIAGONAL
        )

    VisualAestheticsEngine.add_bokeh_lights(
        eng, count=18, colors=[kit.accent, kit.secondary], seed=22
    )
    VisualAestheticsEngine.add_vignette(eng, strength=0.55)

    # Header
    eng.add_rectangle(0, 0, 1080, 100, fill=kit.primary, alpha=0.9)
    eng.add_text(
        "আমন্ত্রণ জানাচ্ছি", x='center', y=28,
        font_alias=kit.font_body,
        font_size=FONT_SCALE["overline"], color=kit.accent
    )

    eng.add_text(
        event_name, x='center', y=200,
        font_alias=kit.font_heading,
        font_size=FONT_SCALE["h1"], color=kit.text_primary,
        max_width=900, shadow=True, line_spacing=16
    )
    eng.add_accent_line(
        460, color=kit.accent,
        width_pct=0.4, thickness=5
    )

    # Info cards
    info_items = [
        ("📅  তারিখ",  date),
        ("⏰  সময়",    time),
        ("📍  স্থান",  venue),
        ("🎟️  টিকিট", ticket),
    ]
    card_y = 500
    for icon_label, value in info_items:
        eng.add_rectangle(
            120, card_y, 960, card_y + 100,
            fill=kit.primary, radius=16, alpha=0.75
        )
        eng.add_text(
            icon_label, x=160, y=card_y + 18,
            font_alias=kit.font_body,
            font_size=FONT_SCALE["caption"], color=kit.accent
        )
        eng.add_text(
            value, x=160, y=card_y + 50,
            font_alias=kit.font_body,
            font_size=FONT_SCALE["body_lg"], color=kit.text_primary
        )
        card_y += 120

    # Footer bar
    eng.add_rectangle(0, 1270, 1080, 1350, fill=kit.accent, alpha=1.0)
    eng.add_text(
        "সবাইকে স্বাগতম • সবার জন্য উন্মুক্ত",
        x='center', y=1294,
        font_alias=kit.font_body,
        font_size=FONT_SCALE["body"],
        color=auto_text_color(kit.accent)
    )
    return eng


@TemplateEngine.register("minimal_card")
def minimal_card(
    name:        str  = "আহমেদ রাহেল",
    title_role:  str  = "সফটওয়্যার ইঞ্জিনিয়ার",
    company:     str  = "TechCorp Bangladesh",
    email:       str  = "rahel@techcorp.com",
    phone:       str  = "+880 1700-000000",
    website:     str  = "www.techcorp.com",
    kit_name:    str  = "ElegantMinimal",
    avatar_bytes: Optional[bytes] = None,
    qr_link:     Optional[str]    = None,
) -> GraphicEngine:
    kit = brand_manager.get_or_default(kit_name)
    eng = GraphicEngine(1050, 600)

    eng.create_solid_background(kit.background)

    # Left accent strip
    eng.add_rectangle(0, 0, 12, 600, fill=kit.accent, alpha=1.0)

    # Right section decoration
    eng.add_circle(900, 300, 300, fill=kit.primary, alpha=0.15)
    eng.add_circle(900, 300, 200, fill=kit.primary, alpha=0.12)

    # Avatar
    if avatar_bytes:
        eng.paste_image(
            avatar_bytes, x=80, y='center',
            width=200, height=200,
            shape=ImageShape.CIRCLE,
            outline_color=kit.accent, outline_width=4
        )
        name_x = 320
    else:
        name_x = 80

    # Name and role
    eng.add_text(
        name, x=name_x, y=130,
        font_alias=kit.font_heading,
        font_size=FONT_SCALE["h2"], color=kit.text_primary
    )
    eng.add_text(
        title_role, x=name_x, y=220,
        font_alias="english_italic",
        font_size=FONT_SCALE["body_lg"], color=kit.accent
    )
    eng.add_text(
        company, x=name_x, y=268,
        font_alias=kit.font_body,
        font_size=FONT_SCALE["body"], color=kit.text_secondary
    )
    eng.add_divider_line(
        320, color=kit.accent,
        alpha=0.3, margin=name_x
    )

    # Contact info
    contact_y = 345
    for icon, info in [("✉", email), ("📞", phone), ("🌐", website)]:
        eng.add_text(
            f"{icon}  {info}", x=name_x, y=contact_y,
            font_alias="english_regular",
            font_size=FONT_SCALE["body_sm"], color=kit.text_secondary
        )
        contact_y += 50

    # QR code
    if qr_link:
        eng.add_qr_code(
            qr_link, x=840, y=120, size=180,
            fill_color=kit.text_primary,
            back_color=kit.background
        )

    return eng


# ── Status ────────────────────────────────────────────────────
templates = TemplateEngine.list_templates()
print("╔══════════════════════════════════════════════════╗")
print("║      CELL 09 — Template Engine ✅               ║")
print("╠══════════════════════════════════════════════════╣")
for i, t in enumerate(templates, 1):
    print(f"║  {i:02d}. {t:<42}║")
print("╠══════════════════════════════════════════════════╣")
print("║  ✅ add_divider_line — AttributeError fixed    ║")
print("╚══════════════════════════════════════════════════╝")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 10 — AUTOMATION ENGINE
# CONFIG.CHUNK_SIZE AttributeError fix
# Bulk generation, CSV import, ZIP export, Report
# ══════════════════════════════════════════════════════════════

@dataclass
class BulkJob:
    """একটি bulk generation job এর সব তথ্য"""
    template:    str
    params:      Dict[str, Any]
    output_name: str  = ""
    platform:    Optional[str] = None
    fmt:         str  = "PNG"
    status:      str  = "pending"   # pending / done / failed
    output_path: str  = ""
    error_msg:   str  = ""
    duration_s:  float = 0.0


@dataclass
class BulkReport:
    """Bulk generation report"""
    total:     int   = 0
    success:   int   = 0
    failed:    int   = 0
    skipped:   int   = 0
    duration_s: float = 0.0
    zip_path:  str   = ""
    jobs:      List[BulkJob] = field(default_factory=list)

    @property
    def success_rate(self) -> float:
        return (self.success / self.total * 100) if self.total > 0 else 0.0

    def print_summary(self):
        print("╔══════════════════════════════════════════════════╗")
        print("║          Bulk Generation Report                  ║")
        print("╠══════════════════════════════════════════════════╣")
        print(f"║  Total    : {self.total:<3}                              ║")
        print(f"║  Success  : {self.success:<3}  ({self.success_rate:.1f}%)                   ║")
        print(f"║  Failed   : {self.failed:<3}                              ║")
        print(f"║  Duration : {self.duration_s:.1f}s                            ║")
        if self.zip_path:
            kb = os.path.getsize(self.zip_path)//1024
            print(f"║  ZIP      : {kb}KB                               ║")
        print("╚══════════════════════════════════════════════════╝")

    def to_json(self) -> str:
        d = {
            "total": self.total,
            "success": self.success,
            "failed": self.failed,
            "skipped": self.skipped,
            "success_rate": round(self.success_rate, 2),
            "duration_s": round(self.duration_s, 2),
            "zip_path": self.zip_path,
            "jobs": [
                {
                    "output_name": j.output_name,
                    "template": j.template,
                    "status": j.status,
                    "output_path": j.output_path,
                    "error_msg": j.error_msg,
                    "duration_s": round(j.duration_s, 3),
                }
                for j in self.jobs
            ]
        }
        return json.dumps(d, indent=2, ensure_ascii=False)


class AutomationEngine:
    """
    Enterprise Automation ও Bulk Generation ইঞ্জিন

    ✅ CONFIG.CHUNK_SIZE fix
    ✅ Bulk generate from list
    ✅ CSV import
    ✅ Progress bar
    ✅ ZIP export
    ✅ JSON report
    ✅ Retry logic
    ✅ Platform-aware export
    """

    def __init__(
        self,
        template:          str,
        jobs:              List[Dict[str, Any]],
        output_dir:        Optional[str] = None,
        effects:           Optional[Dict] = None,
        max_workers:       int  = 2,
        chunk_size:        int  = CONFIG.CHUNK_SIZE,   # ✅ fix — field added in Config
        create_zip:        bool = True,
        create_report:     bool = True,
        platform:          Optional[str] = None,
        fmt:               str  = "PNG",
        prefix:            str  = CONFIG.OUTPUT_PREFIX,
        retry_count:       int  = CONFIG.RETRY_COUNT,
        timestamp_files:   bool = CONFIG.TIMESTAMP_FILES,
    ):
        self.template        = template
        self.jobs            = jobs
        self.output_dir      = output_dir or os.path.join(
            CONFIG.BULK_DIR, template)
        self.effects         = effects or {}
        self.max_workers     = max_workers
        self.chunk_size      = chunk_size
        self.create_zip      = create_zip
        self.create_report   = create_report
        self.platform        = platform
        self.fmt             = fmt
        self.prefix          = prefix
        self.retry_count     = retry_count
        self.timestamp_files = timestamp_files

        os.makedirs(self.output_dir, exist_ok=True)

    def _build_single(
        self, job_dict: Dict[str, Any], index: int
    ) -> BulkJob:
        """একটি job build করুন"""
        params      = {**job_dict}
        output_name = params.pop("__name__", None) or f"{self.prefix}{index:04d}"
        platform    = params.pop("__platform__", self.platform)

        if self.timestamp_files:
            ts = datetime.now().strftime("%H%M%S")
            fname = f"{output_name}_{ts}.{self.fmt.lower()}"
        else:
            fname = f"{output_name}.{self.fmt.lower()}"

        # Merge effects
        if self.effects:
            params.setdefault("effects", {}).update(self.effects)

        job = BulkJob(
            template=self.template,
            params=params,
            output_name=output_name,
            platform=platform,
            fmt=self.fmt,
        )

        t0 = time.time()
        last_err = ""
        for attempt in range(1, self.retry_count + 1):
            try:
                eng = TemplateEngine.build(
                    self.template, params,
                    show=False, save=False
                )

                # Platform export
                if platform and platform in PLATFORM_SIZES:
                    w, h = PLATFORM_SIZES[platform]
                    eng  = ExportEngine.smart_resize(eng, w, h)

                path = eng.save(
                    fname, fmt=self.fmt,
                    directory=self.output_dir
                )
                job.output_path = path
                job.status      = "done"
                break

            except Exception as e:
                last_err = str(e)
                if attempt < self.retry_count:
                    time.sleep(0.1 * attempt)

        if job.status != "done":
            job.status    = "failed"
            job.error_msg = last_err

        job.duration_s = time.time() - t0
        return job

    def run(self) -> BulkReport:
        """সব job চালান"""
        report = BulkReport(total=len(self.jobs))
        t0     = time.time()

        print(f"\n  🚀 Bulk Generation: {self.template}")
        print(f"  {'─'*48}")
        print(f"  📋 Jobs     : {len(self.jobs)}")
        print(f"  📁 Output   : {self.output_dir}")
        print(f"  🔢 Chunk    : {self.chunk_size}")
        print(f"  📐 Platform : {self.platform or 'Original'}")
        print(f"  {'─'*48}")

        bar = tqdm(
            enumerate(self.jobs, 1),
            total=len(self.jobs),
            desc="  ⚡ Generating"
        )

        for i, job_dict in bar:
            job = self._build_single(job_dict, i)
            report.jobs.append(job)
            if job.status == "done":
                report.success += 1
            else:
                report.failed += 1
                print(f"\n  ❌ Job {i}: {job.error_msg[:80]}")
            bar.set_postfix({
                "ok": report.success,
                "fail": report.failed,
                "last": job.output_name[:12],
            })

        report.duration_s = time.time() - t0

        # ZIP
        if self.create_zip and report.success > 0:
            ts       = datetime.now().strftime("%Y%m%d_%H%M%S")
            zip_name = f"{self.template}_{ts}.zip"
            zip_path = os.path.join(CONFIG.EXPORT_DIR, zip_name)
            os.makedirs(CONFIG.EXPORT_DIR, exist_ok=True)
            # ✅ Syntax fix — with block sঠিকভাবে বন্ধ
            with zipfile.ZipFile(
                zip_path,
                mode="w",
                compression=zipfile.ZIP_DEFLATED,
                compresslevel=CONFIG.ZIP_COMPRESSION,
            ) as zf:
                for j in report.jobs:
                    if j.status == "done" and os.path.exists(j.output_path):
                        zf.write(j.output_path, os.path.basename(j.output_path))
            report.zip_path = zip_path

        # Report
        if self.create_report:
            ts          = datetime.now().strftime("%Y%m%d_%H%M%S")
            report_path = os.path.join(
                self.output_dir, f"report_{ts}.json"
            )
            with open(report_path, "w", encoding="utf-8") as f:
                f.write(report.to_json())
            print(f"  📊 Report: {report_path}")

        report.print_summary()
        return report

    # ── CSV Import ────────────────────────────────────────────
    @staticmethod
    def from_csv(
        csv_path:  str,
        template:  str,
        output_dir: Optional[str] = None,
        **kwargs
    ) -> 'AutomationEngine':
        """CSV ফাইল থেকে bulk jobs তৈরি করুন"""
        jobs: List[Dict[str, Any]] = []
        with open(csv_path, "r", encoding="utf-8-sig", newline="") as f:
            reader = csv.DictReader(f)
            for row in reader:
                # খালি string → None
                clean = {
                    k: (v if v.strip() else None)
                    for k, v in row.items()
                }
                jobs.append(clean)
        print(f"  📂 CSV loaded: {len(jobs)} rows from {csv_path}")
        return AutomationEngine(template, jobs, output_dir, **kwargs)

    # ── Quick Batch ───────────────────────────────────────────
    @staticmethod
    def quick_batch(
        template: str,
        data:     List[Dict[str, Any]],
        **kwargs
    ) -> BulkReport:
        """একলাইনে batch generation"""
        engine = AutomationEngine(template, data, **kwargs)
        return engine.run()


# ── Status ────────────────────────────────────────────────────
print("╔══════════════════════════════════════════════════╗")
print("║      CELL 10 — Automation Engine ✅             ║")
print("╠══════════════════════════════════════════════════╣")
print(f"║  CONFIG.CHUNK_SIZE  : {CONFIG.CHUNK_SIZE:<3} ✅ (fixed)           ║")
print(f"║  CONFIG.MAX_BULK    : {CONFIG.MAX_BULK:<3}                    ║")
print(f"║  CONFIG.RETRY_COUNT : {CONFIG.RETRY_COUNT:<3}                    ║")
print("║  ✅ ZipFile() syntax — parenthesis fixed        ║")
print("║  ✅ BulkJob / BulkReport dataclasses            ║")
print("║  ✅ Retry logic                                 ║")
print("║  ✅ CSV import                                  ║")
print("║  ✅ ZIP + JSON report                           ║")
print("║  ✅ quick_batch() helper                        ║")
print("╚══════════════════════════════════════════════════╝")

In [ ]:
# ══════════════════════════════════════════════════════════════
# CELL 11 — INTERACTIVE DASHBOARD
# ipywidgets UI — Template Builder, Brand Kit, Export, Bulk
# ══════════════════════════════════════════════════════════════

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import traceback

# ══════════════════════════════════════════════════════════════
# SECTION A — STYLE HELPERS
# ══════════════════════════════════════���═══════════════════════

_CSS = """
<style>
.cc-dashboard { font-family: 'Segoe UI', sans-serif; }
.cc-header    { background: linear-gradient(135deg,#0F0C29,#302B63,#24243e);
                color:#fff; padding:18px 24px; border-radius:12px;
                margin-bottom:12px; }
.cc-section   { background:#1e1e2e; border-radius:10px;
                padding:14px 18px; margin:8px 0;
                border-left:4px solid #6366F1; }
.cc-success   { color:#38EF7D; font-weight:bold; }
.cc-error     { color:#E63946; font-weight:bold; }
.cc-info      { color:#94A3B8; font-size:0.88em; }
</style>
"""

def _html(content: str) -> widgets.HTML:
    return widgets.HTML(value=content)

def _label(text: str, color: str = "#94A3B8") -> widgets.HTML:
    return widgets.HTML(
        f"<span style='color:{color};font-size:0.85em;"
        f"font-weight:600;letter-spacing:.5px'>{text}</span>"
    )

def _section_title(text: str) -> widgets.HTML:
    return widgets.HTML(
        f"<div style='color:#6366F1;font-weight:700;"
        f"font-size:1.05em;padding:4px 0 8px'>"
        f"▸ {text}</div>"
    )

def _btn(desc: str, style: str = "primary",
         icon: str = "") -> widgets.Button:
    b = widgets.Button(
        description=desc,
        button_style=style,
        icon=icon,
        layout=widgets.Layout(
            height="38px", margin="4px 6px 4px 0"
        )
    )
    return b

def _text(placeholder: str, value: str = "",
          width: str = "380px") -> widgets.Text:
    return widgets.Text(
        value=value,
        placeholder=placeholder,
        layout=widgets.Layout(width=width)
    )

def _textarea(placeholder: str, value: str = "",
              rows: int = 3, width: str = "380px") -> widgets.Textarea:
    return widgets.Textarea(
        value=value,
        placeholder=placeholder,
        rows=rows,
        layout=widgets.Layout(width=width)
    )

def _dropdown(options, value=None,
              width: str = "240px") -> widgets.Dropdown:
    return widgets.Dropdown(
        options=options,
        value=value or options[0],
        layout=widgets.Layout(width=width)
    )

def _int_slider(val, lo, hi,
                desc: str = "", width: str = "300px") -> widgets.IntSlider:
    return widgets.IntSlider(
        value=val, min=lo, max=hi,
        description=desc,
        style={"description_width": "initial"},
        layout=widgets.Layout(width=width)
    )

def _float_slider(val, lo, hi, step=0.05,
                  desc: str = "", width: str = "300px") -> widgets.FloatSlider:
    return widgets.FloatSlider(
        value=val, min=lo, max=hi, step=step,
        description=desc,
        style={"description_width": "initial"},
        layout=widgets.Layout(width=width)
    )

def _checkbox(desc: str, value: bool = False) -> widgets.Checkbox:
    return widgets.Checkbox(
        value=value,
        description=desc,
        indent=False,
        style={"description_width": "initial"}
    )

def _upload(desc: str = "📁 Upload Image",
            accept: str = "image/*") -> widgets.FileUpload:
    return widgets.FileUpload(
        description=desc, accept=accept, multiple=False,
        layout=widgets.Layout(margin="4px 0")
    )

def _divider() -> widgets.HTML:
    return widgets.HTML(
        "<hr style='border:none;border-top:1px solid #334155;"
        "margin:10px 0'>"
    )

def _out(height: str = "auto") -> widgets.Output:
    return widgets.Output(
        layout=widgets.Layout(
            border="1px solid #334155",
            border_radius="8px",
            padding="10px",
            min_height=height,
            background_color="#0f172a",
        )
    )

def _get_upload_bytes(uploader: widgets.FileUpload) -> Optional[bytes]:
    if not uploader.value:
        return None
    try:
        item = list(uploader.value.values())[0]
        return bytes(item["content"])
    except Exception:
        return None


# ══════════════════════════════════════════════════════════════
# SECTION B — TAB 01: TEMPLATE BUILDER
# ══════════════════════════════════════════════════════════════

def _build_template_tab() -> widgets.VBox:
    """Template Builder Tab"""

    # ── Controls ──────────────────────────────────────────��───
    tpl_dropdown = _dropdown(
        TemplateEngine.list_templates(),
        width="260px"
    )
    kit_dropdown = _dropdown(
        brand_manager.list_kits(),
        value="ModernTech",
        width="220px"
    )
    effects_dropdown = _dropdown(
        list(EFFECT_PRESETS.keys()),
        value="minimal",
        width="160px"
    )

    # Dynamic param fields — template ভেদে আলাদা
    param_box = widgets.VBox(layout=widgets.Layout(margin="8px 0"))
    out_preview = _out("320px")
    out_log     = _out("80px")

    # ── Param definitions per template ────────────────────────
    TEMPLATE_PARAMS = {
        "product_promo": [
            ("headline",    "headline",    "নতুন পণ্য এসেছে!",       "text"),
            ("subheadline", "subheadline", "অবিশ্বাস্য মূল্যে",       "text"),
            ("price",       "price",       "৳ ৯৯৯",                   "text"),
            ("cta",         "cta",         "এখনই অর্ডার করুন",        "text"),
            ("qr_link",     "qr_link",     "https://example.com",     "text"),
        ],
        "webinar_flyer": [
            ("title",    "title",    "ডিজিটাল মার্কেটিং ওয়েবিনার", "text"),
            ("subtitle", "subtitle", "আপনার ব্যবসাকে এগিয়ে নিন",   "text"),
            ("date",     "date",     "২০ ফেব্রুয়ারি ২০২৬",          "text"),
            ("time",     "time",     "সন্ধ্যা ৮:০০ টা",              "text"),
            ("speaker",  "speaker",  "মোহাম্মদ রাহেল",              "text"),
            ("platform", "platform", "Zoom • Facebook Live",         "text"),
            ("qr_link",  "qr_link",  "https://zoom.us/j/123",        "text"),
        ],
        "instagram_story": [
            ("headline",  "headline",  "আজকের বিশেষ অফার",   "text"),
            ("body_text", "body_text", "সীমিত সময়ের জন্য ছাড়", "text"),
            ("cta",       "cta",       "স্ক্রোল আপ করুন ↑",  "text"),
        ],
        "sale_announcement": [
            ("percent",  "percent",  "৫০%",          "text"),
            ("label",    "label",    "ছাড়!",         "text"),
            ("subtitle", "subtitle", "সীমিত সময়",   "text"),
            ("code",     "code",     "SALE50",        "text"),
            ("validity", "validity", "৩১ মার্চ পর্যন্ত", "text"),
        ],
        "quote_card": [
            ("quote",    "quote",    "স্বপ্ন দেখুন, পরিশ্রম করুন।", "textarea"),
            ("author",   "author",   "— অজানা",                     "text"),
            ("category", "category", "অনুপ্রেরণা",                  "text"),
        ],
        "youtube_thumbnail": [
            ("title",     "title",     "এই ভিডিওতে সব শিখুন!", "text"),
            ("label",     "label",     "TUTORIAL",              "text"),
            ("presenter", "presenter", "",                      "text"),
        ],
        "facebook_cover": [
            ("brand_name", "brand_name", "আমার ব্র্যান্ড",   "text"),
            ("tagline",    "tagline",    "সেরা মানের পণ্য",   "text"),
            ("website",    "website",   "www.example.com",    "text"),
        ],
        "bengali_special": [
            ("headline",    "headline",    "বিশেষ ঘোষণা",             "text"),
            ("subheadline", "subheadline", "নতুন সেবা চালু হলো",       "text"),
            ("body",        "body",        "আজ থেকেই উপভোগ করুন।",    "textarea"),
            ("footer",      "footer",      "যোগাযোগ: ০১৭০০-০০০০০০",   "text"),
        ],
        "event_poster": [
            ("event_name", "event_name", "বার্ষিক সাংস্কৃতিক উৎসব", "text"),
            ("date",       "date",       "১৫ মার্চ ২০২৬",             "text"),
            ("time",       "time",       "বিকেল ৪:০০ টা",             "text"),
            ("venue",      "venue",      "জাতীয় শিল্পকলা একাডেমি",  "text"),
            ("ticket",     "ticket",     "প্রবেশ মূল্য: ৳ ২০০",      "text"),
        ],
        "minimal_card": [
            ("name",       "name",       "আহমেদ রাহেল",           "text"),
            ("title_role", "title_role", "সফটওয়্যার ইঞ্জিনিয়ার", "text"),
            ("company",    "company",    "TechCorp Bangladesh",   "text"),
            ("email",      "email",      "rahel@techcorp.com",    "text"),
            ("phone",      "phone",      "+880 1700-000000",      "text"),
            ("website",    "website",    "www.techcorp.com",      "text"),
            ("qr_link",    "qr_link",    "https://techcorp.com",  "text"),
        ],
    }

    # Image uploaders (shared)
    up_bg      = _upload("🖼️ Background")
    up_product = _upload("📦 Product/Image")
    up_face    = _upload("👤 Face/Avatar")

    # Param widget registry
    _param_widgets: Dict[str, widgets.Widget] = {}

    def _rebuild_params(change=None):
        tpl = tpl_dropdown.value
        _param_widgets.clear()
        rows = []
        params = TEMPLATE_PARAMS.get(tpl, [])
        for label_text, key, placeholder, wtype in params:
            lbl = _label(label_text)
            if wtype == "textarea":
                w = _textarea(placeholder, placeholder, rows=2, width="360px")
            else:
                w = _text(placeholder, placeholder, width="360px")
            _param_widgets[key] = w
            rows.append(widgets.VBox([lbl, w],
                        layout=widgets.Layout(margin="4px 0")))
        param_box.children = rows

    tpl_dropdown.observe(_rebuild_params, names="value")
    _rebuild_params()  # Initial build

    # ── Build button ──────────────────────────────────────────
    btn_build  = _btn("▶  Build", "primary",  "play")
    btn_save   = _btn("💾 Save",  "success",  "save")
    btn_export = _btn("📤 Export All Platforms", "info", "upload")

    _last_engine: Dict[str, Any] = {"eng": None}

    def on_build(b):
        out_preview.clear_output()
        out_log.clear_output()
        tpl     = tpl_dropdown.value
        kit     = kit_dropdown.value
        effect  = effects_dropdown.value

        params = {"kit_name": kit}
        for key, w in _param_widgets.items():
            v = w.value.strip() if hasattr(w, 'value') else ""
            if v:
                params[key] = v

        # Optional image bytes
        bg_bytes      = _get_upload_bytes(up_bg)
        product_bytes = _get_upload_bytes(up_product)
        face_bytes    = _get_upload_bytes(up_face)

        if bg_bytes:
            params["bg_image_bytes"] = bg_bytes
        if product_bytes:
            # template ভেদে key আলাদা
            params["image_bytes"]   = product_bytes
            params["product_bytes"] = product_bytes
            params["host_bytes"]    = product_bytes
            params["logo_bytes"]    = product_bytes
            params["avatar_bytes"]  = product_bytes
            params["face_bytes"]    = product_bytes

        with out_log:
            print(f"⚙️  Building: {tpl}  |  kit: {kit}  |  fx: {effect}")

        try:
            eng = TemplateEngine.build(
                tpl, params, show=False, save=False
            )
            if effect != "none":
                VisualAestheticsEngine.apply_preset(eng, effect)

            _last_engine["eng"] = eng

            with out_preview:
                eng.show(max_width=520)

            with out_log:
                print(f"<span class='cc-success'>✅ Built successfully!</span>")

        except Exception as e:
            with out_log:
                print(f"❌ Error: {e}")
                traceback.print_exc()

    def on_save(b):
        eng = _last_engine.get("eng")
        if eng is None:
            with out_log:
                print("⚠️  先 Build করুন!")
            return
        ts   = datetime.now().strftime("%Y%m%d_%H%M%S")
        name = f"{tpl_dropdown.value}_{ts}.png"
        path = eng.save(name)
        with out_log:
            print(f"💾 Saved: {path}")

    def on_export(b):
        eng = _last_engine.get("eng")
        if eng is None:
            with out_log:
                print("⚠️  先 Build করুন!")
            return
        with out_log:
            print("📤 Exporting all social platforms...")
        try:
            results = ExportEngine.export_platform_group(
                eng, group="all_social",
                prefix=f"{tpl_dropdown.value}_",
                fmt="PNG", create_zip=True
            )
            with out_log:
                print(f"✅ {len(results)-1} platforms exported")
                if "__zip__" in results:
                    kb = os.path.getsize(results["__zip__"])//1024
                    print(f"📦 ZIP: {results['__zip__']} ({kb}KB)")
        except Exception as e:
            with out_log:
                print(f"❌ {e}")

    btn_build.on_click(on_build)
    btn_save.on_click(on_save)
    btn_export.on_click(on_export)

    # ── Layout ────────────────────────────────────────────────
    left_col = widgets.VBox([
        _section_title("Template & Brand"),
        widgets.HBox([
            widgets.VBox([_label("Template"), tpl_dropdown]),
            widgets.VBox([_label("Brand Kit"), kit_dropdown]),
            widgets.VBox([_label("Effects"), effects_dropdown]),
        ], layout=widgets.Layout(gap="12px", flex_wrap="wrap")),
        _divider(),
        _section_title("Parameters"),
        param_box,
        _divider(),
        _section_title("Images (Optional)"),
        widgets.HBox([up_bg, up_product, up_face],
                     layout=widgets.Layout(gap="8px", flex_wrap="wrap")),
        _divider(),
        widgets.HBox([btn_build, btn_save, btn_export]),
        out_log,
    ], layout=widgets.Layout(width="560px", padding="8px"))

    right_col = widgets.VBox([
        _section_title("Preview"),
        out_preview,
    ], layout=widgets.Layout(
        width="560px", padding="8px"))

    return widgets.HBox(
        [left_col, right_col],
        layout=widgets.Layout(gap="16px", flex_wrap="wrap")
    )


# ══════════════════════════════════════════════════════════════
# SECTION C — TAB 02: BRAND KIT MANAGER
# ══════════════════════════════════════════════════════════════

def _build_brand_tab() -> widgets.VBox:
    kit_sel  = _dropdown(brand_manager.list_kits(), width="220px")
    out_info = _out("160px")
    out_prev = _out("280px")

    # Custom kit fields
    f_name   = _text("Kit name", "MyBrand", width="200px")
    f_pri    = _text("Primary",    "#0F172A", width="130px")
    f_sec    = _text("Secondary",  "#1E293B", width="130px")
    f_acc    = _text("Accent",     "#6366F1", width="130px")
    f_bg     = _text("Background", "#020617", width="130px")

    btn_view   = _btn("👁️ View Kit",    "info")
    btn_create = _btn("✨ Create",       "success")
    btn_test   = _btn("🧪 Preview Demo", "primary")

    def on_view(b):
        out_info.clear_output()
        kit = brand_manager.get_or_default(kit_sel.value)
        with out_info:
            kit.preview()
            cr = BrandEngine.check_contrast(
                kit.text_primary, kit.background)
            print(f"\n  Contrast ratio: {cr['ratio']}  "
                  f"Grade: {cr['grade']}")

    def on_create(b):
        out_info.clear_output()
        try:
            kit = brand_manager.create_custom(
                name=f_name.value.strip() or "CustomKit",
                primary=f_pri.value.strip(),
                accent=f_acc.value.strip(),
                background=f_bg.value.strip(),
            )
            kit_sel.options = brand_manager.list_kits()
            kit_sel.value   = kit.name
            with out_info:
                print(f"✅ Created: {kit.name}")
                kit.preview()
        except Exception as e:
            with out_info:
                print(f"❌ {e}")

    def on_test(b):
        out_prev.clear_output()
        kit = brand_manager.get_or_default(kit_sel.value)
        try:
            with GraphicEngine(600, 300) as eng:
                be = BrandEngine(kit_sel.value)
                be.apply_to(eng)
                VisualAestheticsEngine.add_bokeh_lights(
                    eng, count=8,
                    colors=[kit.accent, kit.primary], seed=1
                )
                eng.add_text(
                    kit.name, x='center', y=60,
                    font_alias=kit.font_heading,
                    font_size=60, color=kit.text_primary
                )
                eng.add_accent_line(
                    155, color=kit.accent,
                    width_pct=0.3, thickness=4
                )
                eng.add_text(
                    f"{kit.primary}  {kit.accent}  {kit.background}",
                    x='center', y=175,
                    font_alias="english_regular",
                    font_size=22, color=kit.text_secondary
                )
                with out_prev:
                    eng.show(max_width=520)
        except Exception as e:
            with out_prev:
                print(f"❌ {e}")
                traceback.print_exc()

    btn_view.on_click(on_view)
    btn_create.on_click(on_create)
    btn_test.on_click(on_test)

    return widgets.VBox([
        widgets.HBox([
            widgets.VBox([
                _section_title("Select & View Brand Kit"),
                widgets.HBox([kit_sel, btn_view, btn_test],
                             layout=widgets.Layout(gap="8px")),
                out_info,
                _divider(),
                _section_title("Create Custom Kit"),
                widgets.HBox([f_name, f_pri, f_acc, f_bg],
                             layout=widgets.Layout(
                                 gap="8px", flex_wrap="wrap")),
                btn_create,
            ], layout=widgets.Layout(width="540px", padding="8px")),
            widgets.VBox([
                _section_title("Kit Preview"),
                out_prev,
            ], layout=widgets.Layout(width="560px", padding="8px")),
        ], layout=widgets.Layout(gap="16px", flex_wrap="wrap")),
    ])


# ══════════════════════════════════════════════════════════════
# SECTION D — TAB 03: EXPORT CENTER
# ══════════════════════════════════════════════════════════════

def _build_export_tab() -> widgets.VBox:
    """Export all platform sizes"""
    tpl_d    = _dropdown(TemplateEngine.list_templates(), width="200px")
    kit_d    = _dropdown(brand_manager.list_kits(),       width="200px")
    grp_d    = _dropdown(list(PLATFORM_GROUPS.keys()),    width="200px")
    fmt_d    = _dropdown(["PNG","JPEG","WEBP"],           width="120px")
    zip_cb   = _checkbox("ZIP ফাইল তৈরি করুন", True)
    up_img   = _upload("🖼️ Background Image")
    out_log  = _out("220px")

    # Platform size viewer
    plat_d   = _dropdown(
        sorted(PLATFORM_SIZES.keys()), width="240px")
    out_size = _out("60px")

    btn_go   = _btn("🚀 Export Group",   "primary")
    btn_info = _btn("ℹ️ Platform Info",  "info")

    def on_info(b):
        out_size.clear_output()
        w, h = PLATFORM_SIZES[plat_d.value]
        with out_size:
            print(f"📐 {plat_d.value}: {w} × {h} px  "
                  f"({w*h//1_000_000:.1f}MP)")

    def on_export(b):
        out_log.clear_output()
        kit = brand_manager.get_or_default(kit_d.value)
        try:
            eng = GraphicEngine(1080, 1080)
            BrandEngine.from_gradient_preset(
                eng, "midnight", GradientDirection.DIAGONAL)
            VisualAestheticsEngine.add_bokeh_lights(
                eng, count=10,
                colors=[kit.accent, kit.primary]
            )
            bg_bytes = _get_upload_bytes(up_img)
            if bg_bytes:
                eng.set_background_image(
                    bg_bytes, blur=8, brightness=0.55,
                    overlay_color=kit.background, overlay_alpha=0.55
                )
            eng.add_text(
                tpl_d.value, x='center', y=460,
                font_alias=kit.font_heading,
                font_size=70, color=kit.text_primary
            )
            VisualAestheticsEngine.add_vignette(eng, strength=0.4)

            with out_log:
                print(f"📤 Exporting group: {grp_d.value}  "
                      f"fmt: {fmt_d.value}")
            results = ExportEngine.export_platform_group(
                eng,
                group=grp_d.value,
                prefix=f"{tpl_d.value}_",
                fmt=fmt_d.value,
                create_zip=zip_cb.value,
            )
            with out_log:
                n = sum(1 for k in results if not k.startswith("__"))
                print(f"✅ {n} platforms exported")
                if "__zip__" in results:
                    kb = os.path.getsize(results["__zip__"])//1024
                    print(f"📦 ZIP ({kb}KB): {results['__zip__']}")

        except Exception as e:
            with out_log:
                print(f"❌ {e}")
                traceback.print_exc()

    btn_go.on_click(on_export)
    btn_info.on_click(on_info)

    return widgets.VBox([
        _section_title("Platform Export Center"),
        widgets.HBox([
            widgets.VBox([_label("Template"), tpl_d]),
            widgets.VBox([_label("Brand Kit"), kit_d]),
            widgets.VBox([_label("Group"), grp_d]),
            widgets.VBox([_label("Format"), fmt_d]),
        ], layout=widgets.Layout(gap="12px", flex_wrap="wrap")),
        widgets.HBox([zip_cb, up_img],
                     layout=widgets.Layout(gap="16px")),
        btn_go,
        out_log,
        _divider(),
        _section_title("Platform Size Reference"),
        widgets.HBox([plat_d, btn_info],
                     layout=widgets.Layout(gap="8px")),
        out_size,
    ], layout=widgets.Layout(padding="12px"))


# ══════════════════════════════════════════════════════════════
# SECTION E — TAB 04: BULK GENERATOR
# ══════════════════════════════════════════════════════════════

def _build_bulk_tab() -> widgets.VBox:
    tpl_d   = _dropdown(TemplateEngine.list_templates(), width="200px")
    kit_d   = _dropdown(brand_manager.list_kits(),       width="200px")
    plat_d  = _dropdown(
        ["(none)"] + sorted(PLATFORM_SIZES.keys()), width="200px")
    fmt_d   = _dropdown(["PNG","JPEG","WEBP"], width="120px")
    zip_cb  = _checkbox("ZIP তৈরি করুন", True)
    rpt_cb  = _checkbox("Report সেভ করুন", True)

    # CSV bulk input
    csv_area = _textarea(
        "headline,subheadline,price\n"
        "পণ্য ১,বিশেষ অফার,৳ ৪৯৯\n"
        "পণ্য ২,সীমিত ছাড়,৳ ৭৯৯",
        rows=6, width="520px"
    )
    out_log = _out("240px")

    btn_run    = _btn("▶ Run Bulk",       "primary")
    btn_sample = _btn("📋 Load Sample",   "info")
    btn_clear  = _btn("🗑️  Clear Log",    "warning")

    SAMPLES = {
        "product_promo": (
            "headline,subheadline,price,cta\n"
            "গরমের স্পেশাল,৩০% ছাড়,৳ ৮৯৯,এখনই কিনুন\n"
            "বর্ষার অফার,সীমিত সময়,৳ ১২৯৯,অর্ডার করুন\n"
            "ঈদ স্পেশাল,বিশেষ মূল্যে,৳ ৬৪৯,কিনতে ক্লিক করুন\n"
        ),
        "quote_card": (
            "quote,author,category\n"
            "পরিশ্রমই সাফল্যের চাবিকাঠি,— অজানা,অনুপ্রেরণা\n"
            "স্বপ্ন বড় দেখুন,— বিল গেটস,উদ্যোগ\n"
            "সময় মূল্যবান সম্পদ,— বেঞ্জামিন ফ্র্যাংকলিন,জ্ঞান\n"
        ),
        "sale_announcement": (
            "percent,label,code,subtitle\n"
            "৪০%,ছাড়!,SAVE40,বর্ষা সেল\n"
            "৬০%,বিশেষ ছাড়!,MEGA60,ক্লিয়ারেন্স সেল\n"
            "২৫%,শীতকালীন অফার!,WINTER25,সীমিত সময়\n"
        ),
        "webinar_flyer": (
            "title,date,time,speaker,platform\n"
            "ডিজিটাল মার্কেটিং,২০ মার্চ,বিকেল ৫টা,রাহেল হোসেন,Zoom\n"
            "ফ্রিল্যান্সিং গাইড,২২ মার্চ,রাত ৯টা,তামিম ইকবাল,Facebook Live\n"
        ),
    }

    def on_sample(b):
        csv_area.value = SAMPLES.get(
            tpl_d.value,
            "headline,subheadline\nটেস্ট পণ্য ১,বিশেষ অফার\nটেস্ট পণ্য ২,সীমিত ছাড়\n"
        )

    def on_clear(b):
        out_log.clear_output()

    def on_run(b):
        out_log.clear_output()
        csv_text = csv_area.value.strip()
        if not csv_text:
            with out_log:
                print("⚠️  CSV data দিন!")
            return

        try:
            reader = csv.DictReader(io.StringIO(csv_text))
            jobs   = []
            for row in reader:
                job = {"kit_name": kit_d.value}
                for k, v in row.items():
                    if v and v.strip():
                        job[k.strip()] = v.strip()
                jobs.append(job)

            if not jobs:
                with out_log:
                    print("⚠️  CSV parse করা যায়নি।")
                return

            plat = plat_d.value
            if plat == "(none)":
                plat = None

            with out_log:
                print(f"⚙️  {len(jobs)} jobs শুরু হচ্ছে...")

            engine = AutomationEngine(
                template=tpl_d.value,
                jobs=jobs,
                platform=plat,
                fmt=fmt_d.value,
                create_zip=zip_cb.value,
                create_report=rpt_cb.value,
            )
            report = engine.run()

            with out_log:
                print(f"\n✅ Done — "
                      f"{report.success}/{report.total} সফল  "
                      f"({report.duration_s:.1f}s)")
                if report.zip_path and os.path.exists(report.zip_path):
                    kb = os.path.getsize(report.zip_path)//1024
                    print(f"📦 ZIP ({kb}KB): {report.zip_path}")

        except Exception as e:
            with out_log:
                print(f"❌ {e}")
                traceback.print_exc()

    btn_sample.on_click(on_sample)
    btn_clear.on_click(on_clear)
    btn_run.on_click(on_run)

    return widgets.VBox([
        _section_title("Bulk Generation Settings"),
        widgets.HBox([
            widgets.VBox([_label("Template"), tpl_d]),
            widgets.VBox([_label("Brand Kit"), kit_d]),
            widgets.VBox([_label("Platform"), plat_d]),
            widgets.VBox([_label("Format"), fmt_d]),
        ], layout=widgets.Layout(gap="12px", flex_wrap="wrap")),
        widgets.HBox([zip_cb, rpt_cb],
                     layout=widgets.Layout(gap="20px")),
        _divider(),
        _section_title("CSV Data  (প্রথম row = column headers)"),
        widgets.HBox([btn_sample, btn_clear]),
        csv_area,
        btn_run,
        out_log,
    ], layout=widgets.Layout(padding="12px"))


# ══════════════════════════════════════════════════════════════
# SECTION F — TAB 05: VISUAL EFFECTS TESTER
# ══════════════════════════════════════════════════════════════

def _build_effects_tab() -> widgets.VBox:
    kit_d        = _dropdown(brand_manager.list_kits(),       width="200px")
    grad_d       = _dropdown(list(GRADIENT_PRESETS.keys()),   width="200px")
    grad_dir_d   = _dropdown(
        [d.value for d in GradientDirection], width="160px")
    texture_d    = _dropdown(
        [t.value for t in TextureType], width="160px")
    tex_strength = _float_slider(0.12, 0.0, 0.5,
                                 desc="Texture",  width="280px")
    vig_str      = _float_slider(0.5,  0.0, 1.0,
                                 desc="Vignette", width="280px")
    bokeh_count  = _int_slider(12, 0, 30,
                               desc="Bokeh",     width="280px")
    grade_d      = _dropdown(
        ["none","cinematic","vibrant","muted","warm","cool","bw"],
        width="160px")
    glass_cb     = _checkbox("Glassmorphism panel", True)
    noise_cb     = _checkbox("Noise overlay", False)

    out_prev = _out("460px")
    btn_apply = _btn("✨ Apply Effects", "primary")
    btn_save  = _btn("💾 Save",         "success")
    _last: Dict = {"eng": None}

    def on_apply(b):
        out_prev.clear_output()
        kit = brand_manager.get_or_default(kit_d.value)
        try:
            eng = GraphicEngine(800, 400)
            # Gradient
            preset = grad_d.value
            c1, c2 = GRADIENT_PRESETS[preset]
            eng.create_gradient_background(
                c1, c2,
                GradientDirection(grad_dir_d.value)
            )

            # Bokeh
            if bokeh_count.value > 0:
                VisualAestheticsEngine.add_bokeh_lights(
                    eng, count=bokeh_count.value,
                    colors=[kit.accent, kit.primary, c1, c2]
                )

            # Texture
            if texture_d.value != "none":
                VisualAestheticsEngine.apply_texture(
                    eng,
                    TextureType(texture_d.value),
                    strength=tex_strength.value
                )

            # Glass panel
            if glass_cb.value:
                VisualAestheticsEngine.add_glassmorphism(
                    eng, 40, 100, 760, 300,
                    blur_radius=14, fill_opacity=0.12,
                    border_color=kit.accent, border_opacity=0.3,
                    radius=20
                )

            # Text
            eng.add_text(
                f"Effects: {preset}  ✦  {grade_d.value}",
                x='center', y=150,
                font_alias="english_bold",
                font_size=38, color="#FFFFFF",
                shadow=True
            )
            eng.add_text(
                f"Bokeh:{bokeh_count.value}  "
                f"Texture:{texture_d.value}  "
                f"Vig:{vig_str.value:.2f}",
                x='center', y=210,
                font_alias="english_regular",
                font_size=24, color="#94A3B8"
            )

            # Vignette
            if vig_str.value > 0:
                VisualAestheticsEngine.add_vignette(
                    eng, strength=vig_str.value)

            # Noise
            if noise_cb.value:
                VisualAestheticsEngine.add_noise(eng, strength=0.06)

            # Color grade
            if grade_d.value != "none":
                VisualAestheticsEngine.apply_color_grade(
                    eng, preset=grade_d.value)

            _last["eng"] = eng
            with out_prev:
                eng.show(max_width=700)

        except Exception as e:
            with out_prev:
                print(f"❌ {e}")
                traceback.print_exc()

    def on_save(b):
        eng = _last.get("eng")
        if eng is None:
            return
        ts   = datetime.now().strftime("%Y%m%d_%H%M%S")
        path = eng.save(f"effects_test_{ts}.png")
        with out_prev:
            print(f"💾 Saved: {path}")

    btn_apply.on_click(on_apply)
    btn_save.on_click(on_save)

    return widgets.VBox([
        widgets.HBox([
            widgets.VBox([
                _section_title("Background"),
                widgets.HBox([
                    widgets.VBox([_label("Gradient Preset"), grad_d]),
                    widgets.VBox([_label("Direction"),       grad_dir_d]),
                    widgets.VBox([_label("Brand Kit"),       kit_d]),
                ], layout=widgets.Layout(gap="12px")),
                _divider(),
                _section_title("Effects"),
                widgets.HBox([
                    widgets.VBox([_label("Color Grade"),  grade_d]),
                    widgets.VBox([_label("Texture Type"), texture_d]),
                ], layout=widgets.Layout(gap="12px")),
                tex_strength,
                vig_str,
                bokeh_count,
                widgets.HBox([glass_cb, noise_cb],
                             layout=widgets.Layout(gap="16px")),
                _divider(),
                widgets.HBox([btn_apply, btn_save]),
            ], layout=widgets.Layout(width="420px", padding="8px")),
            widgets.VBox([
                _section_title("Preview"),
                out_prev,
            ], layout=widgets.Layout(width="720px", padding="8px")),
        ], layout=widgets.Layout(gap="16px", flex_wrap="wrap")),
    ])


# ══════════════════════════════════════════════════════════════
# SECTION G — TAB 06: SYSTEM STATUS
# ══════════════════════════════════════════════════════════════

def _build_status_tab() -> widgets.VBox:
    out = _out("420px")

    btn_check = _btn("🔍 System Check",  "primary")
    btn_fonts = _btn("🔤 Font Status",   "info")
    btn_kits  = _btn("🎨 Brand Kits",    "info")
    btn_dirs  = _btn("📁 Directories",   "warning")
    btn_clear = _btn("🗑️  Clear",        "")

    def on_check(b):
        out.clear_output()
        with out:
            print("╔══════════════════════════════════════════════════╗")
            print("║         ColabCanvas — System Status             ║")
            print("╠══════════════════════════════════════════════════╣")
            print(f"║  NumPy    : {np.__version__:<10}                       ║")
            print(f"║  Pillow   : {Image.__version__:<10}                       ║")
            print(f"║  OpenCV   : {cv2.__version__:<10}  ✅                  ║")
            print(f"║  CV2      : {'✅' if CV2_AVAILABLE    else '❌'}                                ║")
            print(f"║  rembg    : {'✅' if REMBG_AVAILABLE  else '❌ (pip install rembg)'}           ║")
            print(f"║  fpdf2    : {'✅' if FPDF_AVAILABLE   else '❌ (pip install fpdf2)'}           ║")
            print(f"║  sklearn  : {'✅' if SKLEARN_AVAILABLE else '❌ (optional)'}                  ║")
            print("╠══════════════════════════════════════════════════╣")
            print(f"║  Templates : {len(TemplateEngine.list_templates()):<3}                             ║")
            print(f"║  BrandKits : {len(brand_manager.list_kits()):<3}                             ║")
            print(f"║  Fonts OK  : {font_manager.ready_count():<3}/{len(FONT_CATALOG):<3}                        ║")
            print(f"║  Platforms : {len(PLATFORM_SIZES):<3}                             ║")
            print(f"║  Gradients : {len(GRADIENT_PRESETS):<3}                             ║")
            print("╠══════════════════════════════════════════════════╣")
            print(f"║  CONFIG.CHUNK_SIZE  : {CONFIG.CHUNK_SIZE:<3}               ✅   ║")
            print(f"║  CONFIG.MAX_BULK    : {CONFIG.MAX_BULK:<3}               ✅   ║")
            print(f"║  CONFIG.RETRY_COUNT : {CONFIG.RETRY_COUNT:<3}               ✅   ║")
            print("╚══════════════════════════════════════════════════╝")

    def on_fonts(b):
        out.clear_output()
        with out:
            print(f"🔤 Font Status ({font_manager.ready_count()}/{len(FONT_CATALOG)}):")
            print("─" * 44)
            for alias, info in FONT_CATALOG.items():
                path = os.path.join(CONFIG.FONT_DIR, info['file'])
                ok   = os.path.exists(path)
                size = (os.path.getsize(path)//1024) if ok else 0
                icon = "✅" if ok else "❌"
                print(f"  {icon} {alias:<22} "
                      f"{f'{size}KB' if ok else 'missing'}")

    def on_kits(b):
        out.clear_output()
        with out:
            kits = brand_manager.list_kits()
            print(f"🎨 Brand Kits ({len(kits)}):")
            print("─" * 44)
            for name in kits:
                kit = brand_manager.get_or_default(name)
                cr  = BrandEngine.check_contrast(
                    kit.text_primary, kit.background)
                print(f"  {name:<18} "
                      f"accent:{kit.accent}  "
                      f"contrast:{cr['ratio']}")

    def on_dirs(b):
        out.clear_output()
        with out:
            print("📁 Directories:")
            print("─" * 44)
            dirs = {
                "fonts":      CONFIG.FONT_DIR,
                "outputs":    CONFIG.OUTPUT_DIR,
                "exports":    CONFIG.EXPORT_DIR,
                "bulk":       CONFIG.BULK_DIR,
                "brand_kits": CONFIG.BRAND_DIR,
                "assets":     CONFIG.ASSET_DIR,
                "cache":      CONFIG.CACHE_DIR,
                "temp":       CONFIG.TEMP_DIR,
            }
            for label, path in dirs.items():
                exists = os.path.isdir(path)
                files  = len(os.listdir(path)) if exists else 0
                icon   = "✅" if exists else "❌"
                print(f"  {icon} {label:<12} {path:<20} ({files} files)")

    btn_check.on_click(on_check)
    btn_fonts.on_click(on_fonts)
    btn_kits.on_click(on_kits)
    btn_dirs.on_click(on_dirs)
    btn_clear.on_click(lambda b: out.clear_output())

    return widgets.VBox([
        _section_title("System Status & Debug"),
        widgets.HBox(
            [btn_check, btn_fonts, btn_kits, btn_dirs, btn_clear],
            layout=widgets.Layout(gap="8px", flex_wrap="wrap")
        ),
        out,
    ], layout=widgets.Layout(padding="12px"))


# ══════════════════════════════════════════════════════════════
# SECTION H — ASSEMBLE DASHBOARD
# ══════════════════════════════════════════════════════════════

def launch_dashboard():
    """ColabCanvas Interactive Dashboard চালু করুন"""

    display(HTML(_CSS))

    header = widgets.HTML("""
    <div class='cc-header'>
        <h2 style='margin:0;font-size:1.6em'>
            🎨 ColabCanvas Dashboard
        </h2>
        <p style='margin:4px 0 0;color:#94A3B8;font-size:.9em'>
            Template Builder · Brand Kit · Export · Bulk · Effects · System
        </p>
    </div>
    """)

    tab = widgets.Tab(layout=widgets.Layout(width="100%"))
    tab.children = [
        _build_template_tab(),
        _build_brand_tab(),
        _build_export_tab(),
        _build_bulk_tab(),
        _build_effects_tab(),
        _build_status_tab(),
    ]
    for i, title in enumerate([
        "🎨 Template Builder",
        "🏷️ Brand Kit",
        "📤 Export",
        "⚡ Bulk",
        "✨ Effects",
        "🔧 System",
    ]):
        tab.set_title(i, title)

    display(widgets.VBox([header, tab]))


# ── Launch ────────────────────────────────────────────────────
launch_dashboard()